# NB-08 · Stage 8 — Chief Resident skip-distillation

**Project:** HiDMamba, hierarchical distillation for 12-lead ECG (PTB-XL, 100 Hz)
**Runtime:** Kaggle, GPU T4. The student has fewer than one million parameters and never runs a teacher: expect about 10 s per epoch. The whole notebook (pilot, the main model with three seeds, nine ablation and control arms, evaluation, 20+ figures) is planned for 2.5 to 4 hours and resumes by itself if the session is stopped.

Synopsis, Stage 8: *train the standalone Chief Resident (patchifier, single bidirectional Mamba block, classification head) against the cached artefacts of Stage 7 with*

`L_ChiefResident = L_supervised + λ1·L_professor_KD + λ2·L_gate_mimic + λ3·L_RKD`

*During training only, a Cross-Attention Consultation module (query = the student's own representation, key/value = cached Professor embeddings) produces an auxiliary, richer prediction; a self-distillation term trains the standalone, Consultation-free path to match it. At the end the Consultation module and its auxiliary head are discarded; only the patchifier, the Mamba block and the standalone head are kept.*

### What this notebook does
1. **Loads and verifies the Stage-7 cache** (manifest, shapes, checksums on request, row order against `split_idx.npz`, labels against the CSV).
2. **Builds the student** at the largest width that keeps the *deployed* network under one million parameters, and shows the training-only parts separately.
3. **Smoke-tests** every loss and, most importantly, the one structural promise of the design: *the standalone path cannot see the Professors* (its output is unchanged when the cached Professor embeddings are replaced).
4. **Picks the learning rate** with a short pilot (a rate is eligible only if no batch had a non-finite loss).
5. **Trains the ten arms** that answer the questions a reviewer will ask (section 7), with three seeds for the three arms the conclusions rest on.
6. **Evaluates on fold 9 only** (calibration, thresholds, bootstrap intervals, paired differences against every teacher and every ablation, a claims table with the verdict of each claim computed from the data).
7. **Exports the deployable artefact** (`chief_resident_deploy.pt`: patchifier, block and head, nothing else), reloads it from scratch and checks it reproduces the trained model.

### Inputs (attach these Datasets)
| Dataset | Files used |
|---|---|
| `cached-teacher-outputs-v1` (Stage 7, version 3) | `cache/*.npy`, `teacher_cache_manifest.json`, `cache_lib.py`, `dean_lib.py`, `eval_lib.py`, `hod_lib.py` |
| `dean-finetuned-v2` (Stage 3) | `dean_finetuned.pt` (encoder configuration and the input normalisation), `split_idx.npz` |
| `ptbxl-processed-v1` (Stage 1) | `ptbxl_100hz.npy`, `labels_multihot.csv` |

Fold 10 is never read. It is touched once, in Stage 10, on the final chosen model.

### Outputs (`/kaggle/working`, publish as the Dataset `chief-resident-v1`)
| File | Content |
|---|---|
| `chief_resident_deploy.pt`, `cr_lib.py` | the shipped network, with label order, input normalisation, Platt parameters and thresholds |
| `cr_logits_{train,val}.npy`, `cr_emb_{train,val}.npy`, `cr_gate_val.npy` | outputs for Stage 9 and the paper |
| `cr_metrics.json`, `cr_claims.json`, `tables/*.csv` | every number in the results |
| `runs/*_final.pt` | all arms (resumable; delete to retrain) |
| `figures/*.png|pdf` | the figures, 7.2 inches wide (or 3.5 for a single column), 300 dpi |

### Decisions that interpret the written plan
* **Width.** The plan says "single bidirectional Mamba block" and the project's budget is under one million deployed parameters, so the width is the largest multiple of 16 that satisfies that (section 3). The training-only modules are counted separately and are not part of the deployed size.
* **Distillation targets.** Professors have no superclass output, so the superclass columns of `L_professor_KD` come from the HOD committee (the departmental heads that the Professors descend from); the 21 subclass columns come from the Professors. This is the literal reading of the cascade ("chain"). The other teachers (DEAN out-of-fold, DEAN in-sample, HOD committee for all columns, an ensemble) are control arms, not alternatives that replace it.
* **What Stage 7 showed.** Professor and HOD logits on the training records are *in-sample*: training AUC 0.998 against 0.934 on fold 9, a gap of 0.064. DEAN's out-of-fold logits have no such gap (−0.002). Whether in-sample Professor logits are a *useful* target is therefore an empirical question that this notebook answers with controls, and the answer is reported even if it is unfavourable to the hierarchy.
* **Augmentation.** The cached targets belong to the clean record, so only the mild, teacher-invariant augmentation of Stage 5 is used (amplitude, baseline wander, noise).
* **Gate.** The gate head is trained to match the *cross-fitted* routing distribution of Stage 4 (`route_train`), never the in-sample one. It is training-only.
* **Consultation oracle caution.** At validation time the Consultation module reads the cached Professor embeddings of the validation record. Those embeddings come from models that were fitted on other folds, so the auxiliary AUC on fold 9 is honest, but on *training* records the embeddings come from models that saw the label. The auxiliary head can therefore exploit them during training in a way that does not transfer. Section 11 measures how much, and the standalone path is what is evaluated.

## 0. Environment

In [1]:
import os, sys, glob, json, time, math, copy, random, shutil, warnings, platform, importlib
from collections import defaultdict, OrderedDict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from scipy.stats import spearmanr

warnings.filterwarnings('ignore', category=UserWarning)
NB_T0 = time.time()

ON_KAGGLE = os.path.exists('/kaggle/working')
WORK = '/kaggle/working' if ON_KAGGLE else os.path.abspath('./working')
INPUT_ROOT = os.environ.get('NB_INPUT_ROOT', '/kaggle/input')
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
RUN_DIR, FIG_DIR, TAB_DIR = (os.path.join(WORK, d) for d in ('runs', 'figures', 'tables'))
for d in (RUN_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if ON_KAGGLE:
    assert dev.type == 'cuda', 'No GPU. Settings -> Accelerator -> GPU T4.'
USE_AMP = dev.type == 'cuda'
if USE_AMP:
    torch.backends.cudnn.benchmark = True

def amp():
    return torch.autocast('cuda', dtype=torch.float16) if USE_AMP else nullcontext()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if USE_AMP: torch.cuda.manual_seed_all(s)

def find_art(name):
    hits = sorted(glob.glob(f'{WORK}/**/{name}', recursive=True)) + sorted(glob.glob(f'{INPUT_ROOT}/**/{name}', recursive=True))
    hits = [h for h in hits if os.sep + 'runs' + os.sep not in h]
    return hits[0] if hits else None

def find_one(name, hint=''):
    p = find_art(name)
    if p is None:
        raise FileNotFoundError(f'{name} not found. Attach the Dataset {hint or "that contains it"}.')
    return p

def hours_left(max_hours):
    return max_hours - (time.time() - NB_T0) / 3600

VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__}
print(VERSIONS)
print('device', dev, '|', torch.cuda.get_device_name(0) if USE_AMP else 'cpu', '| work', WORK)

{'python': '3.13.15', 'torch': '2.11.0+cu128', 'numpy': '2.1.3', 'pandas': '2.3.3'}
device cuda | Tesla T4 | work /kaggle/working


## 1. Configuration
The weights of the three distillation terms are the plan's (`λ1 = 1`, `λ3 = 0.5` as in Stages 5 and 6) with `λ2 = 0.5` for the gate term, whose KL value is about 0.3 at the start. They are not tuned on fold 9; the ablation tells whether each term earns its place.

* **Optimiser:** AdamW, cosine schedule with two warm-up epochs, weight decay 0.05, gradient clipping 1.0, EMA of the weights (the EMA copy is validated, exported and evaluated), early stopping on the validation macro-AUC of the **standalone** path.
* **Divergence guard** (the lesson of Stage 5, where a learning rate of 1e-3 produced a non-finite loss on one head): a non-finite batch is skipped and counted; twenty-five in a row abort the run. A learning rate is eligible in the pilot only if its run skipped no batch.
* **Consultation** drops each Professor with probability `1 − prof_keep` per record during training.

In [2]:
CFG = dict(
    seed=42, min_pos=10, n_boot=300,
    # student
    d_model=None, param_budget=1_000_000, heads=4, head_drop=0.2, sub_hidden=256, consult_drop=0.1, prof_keep=0.8,
    # optimisation
    epochs=40, batch_size=64, lr=None, lr_default=5e-4, wd=0.05, warmup_epochs=2, lr_floor=0.02, clip=1.0,
    patience=10, ema_decay=0.998, save_every=2, max_hours=11.0, max_skip=25,
    # loss
    label_smooth=0.01, w_sup=1.0, w_kd=1.0, w_gate=0.5, w_rkd=0.5, w_aux=1.0, w_sd=1.0, T=3.0, T_sd=2.0, aug_strength=1.0,
    kd_src='chain',
    # learning-rate pilot (on the full model, one seed)
    run_lr_pilot=True, pilot_lrs=[3e-4, 6e-4, 1e-3], pilot_epochs=8, pilot_margin=0.002,
    # arms and seeds
    seeds=[0, 1, 2], seed_arms=['A_sup', 'D_three_term', 'E_full'],
    # optional analyses
    run_occlusion=True, run_tsne=True, n_tsne=2000, strict=True,
)
print({k: CFG[k] for k in ['param_budget', 'epochs', 'batch_size', 'w_kd', 'w_gate', 'w_rkd', 'w_aux', 'w_sd', 'T', 'T_sd', 'run_lr_pilot', 'seeds']})

{'param_budget': 1000000, 'epochs': 40, 'batch_size': 64, 'w_kd': 1.0, 'w_gate': 0.5, 'w_rkd': 0.5, 'w_aux': 1.0, 'w_sd': 1.0, 'T': 3.0, 'T_sd': 2.0, 'run_lr_pilot': True, 'seeds': [0, 1, 2]}


## 2. Libraries
`cache_lib.py`, `dean_lib.py`, `eval_lib.py` and `hod_lib.py` come from the Stage-7 Dataset (it republishes them), so the metrics, the Mamba block, the scan and the losses are exactly the ones every earlier stage used. `cr_lib.py` is written below; it holds the student, the Consultation module and the deployable network. `hod_lib.fast_scan` replaces the library scan (2.9 times faster on a T4, outputs and gradients checked in Stage 5 and again in the smoke test).

In [3]:
%%writefile cr_lib.py
"""Chief Resident of HiDMamba (stage 8): the student, its training-only modules and the deployable artefact.

Deployed path   : patchifier (Conv1d) + positional embedding -> ONE bidirectional Mamba block -> final norm -> mean pool -> head.
Training only   : gate head (mimics the gating distribution), Consultation (cross-attention over cached Professor embeddings)
                  and the auxiliary head fed by it. None of them is part of ChiefResidentDeploy.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dean_lib import DEANEncoder

TRAIN_ONLY_PREFIXES = ('gate.', 'consult.', 'aux_')


def mlp(d, hidden, n_out, drop):
    return nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, n_out))


class Consultation(nn.Module):
    """Query = the student's own pooled representation z (B,D); keys and values = cached Professor embeddings (B,P,Dp).
    A learned identity vector is added to every key so that the attention knows WHICH Professor it is looking at.
    `keep` (B,P) bool drops Professors at random during training so the module cannot lean on one of them."""

    def __init__(self, d, d_prof, n_prof, heads=4, drop=0.1):
        super().__init__()
        assert d % heads == 0, 'student width must be divisible by the number of attention heads'
        self.h, self.dk = heads, d // heads
        self.kv_norm = nn.LayerNorm(d_prof)
        self.k, self.v = nn.Linear(d_prof, d), nn.Linear(d_prof, d)
        self.prof_id = nn.Parameter(torch.zeros(1, n_prof, d))
        nn.init.trunc_normal_(self.prof_id, std=0.02)
        self.norm_q, self.q, self.o = nn.LayerNorm(d), nn.Linear(d, d), nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)
        self.ffn = mlp(d, 2 * d, d, drop)
        self.drop = nn.Dropout(drop)

    def forward(self, z, P, keep=None):
        B, n, _ = P.shape
        Pn = self.kv_norm(P.float())
        K = (self.k(Pn) + self.prof_id).view(B, n, self.h, self.dk).transpose(1, 2)        # (B,h,n,dk)
        V = self.v(Pn).view(B, n, self.h, self.dk).transpose(1, 2)
        q = self.q(self.norm_q(z)).view(B, 1, self.h, self.dk).transpose(1, 2)            # (B,h,1,dk)
        s = (q @ K.transpose(-1, -2)).float() / math.sqrt(self.dk)                          # (B,h,1,n)
        if keep is not None:
            s = s.masked_fill(~keep[:, None, None, :], -1e4)
        a = torch.softmax(s, dim=-1)
        ctx = (a.to(V.dtype) @ V).transpose(1, 2).reshape(B, -1)
        out = z + self.drop(self.o(ctx))
        out = out + self.drop(self.ffn(self.norm(out)))
        return out, a.squeeze(2)                                                             # (B,h,n)


class ChiefResident(nn.Module):
    def __init__(self, enc_cfg, d_model, n_sup=5, n_sub=21, head_drop=0.2, sub_hidden=256, n_route=5,
                 prof_dim=None, n_prof=21, heads=4, consult_drop=0.1):
        super().__init__()
        c = dict(enc_cfg)
        c.update(d_model=d_model, depth=1, grad_ckpt=False)
        self.enc = DEANEncoder(**c)
        self.n_sup, self.n_sub, self.d = n_sup, n_sub, d_model
        # ---- deployed head
        self.drop = nn.Dropout(head_drop)
        self.sup = nn.Linear(d_model, n_sup)
        self.sub = mlp(d_model, sub_hidden, n_sub, head_drop)
        # ---- training only
        self.gate = nn.Linear(d_model, n_route)
        self.consult = Consultation(d_model, prof_dim, n_prof, heads, consult_drop) if prof_dim else None
        if self.consult is not None:
            self.aux_drop = nn.Dropout(head_drop)
            self.aux_sup = nn.Linear(d_model, n_sup)
            self.aux_sub = mlp(d_model, sub_hidden, n_sub, head_drop)

    def set_prior(self, p):
        b = torch.log(p / (1 - p)).to(self.sup.bias.device)
        with torch.no_grad():
            self.sup.bias.copy_(b[:self.n_sup]); self.sub[-1].bias.copy_(b[self.n_sup:])
            if self.consult is not None:
                self.aux_sup.bias.copy_(b[:self.n_sup]); self.aux_sub[-1].bias.copy_(b[self.n_sup:])

    def forward(self, x, prof=None, keep=None):
        tok = self.enc(x)
        z = tok.float().mean(1)
        h = self.drop(z)
        out = dict(logits=torch.cat([self.sup(h), self.sub(h)], 1), z=z, tok=tok, gate=self.gate(z))
        if prof is not None and self.consult is not None:
            zc, att = self.consult(z, prof, keep)
            h2 = self.aux_drop(zc)
            out['aux'] = torch.cat([self.aux_sup(h2), self.aux_sub(h2)], 1)
            out['att'] = att
        return out

    def deploy_state(self):
        return {k: v for k, v in self.state_dict().items() if k.startswith(('enc.', 'sup.', 'sub.'))}

    def n_deployed(self):
        return sum(p.numel() for n, p in self.named_parameters() if n.startswith(('enc.', 'sup.', 'sub.')))

    def n_training_only(self):
        return sum(p.numel() for n, p in self.named_parameters() if n.startswith(TRAIN_ONLY_PREFIXES))


class ChiefResidentDeploy(nn.Module):
    """What is shipped: patchifier + one bidirectional Mamba block + head. Nothing else."""

    def __init__(self, enc_cfg, d_model, n_sup=5, n_sub=21, head_drop=0.2, sub_hidden=256):
        super().__init__()
        c = dict(enc_cfg)
        c.update(d_model=d_model, depth=1, grad_ckpt=False)
        self.enc = DEANEncoder(**c)
        self.drop = nn.Dropout(head_drop)
        self.sup = nn.Linear(d_model, n_sup)
        self.sub = mlp(d_model, sub_hidden, n_sub, head_drop)

    def forward(self, x, full=False):
        tok = self.enc(x)
        z = tok.float().mean(1)
        h = self.drop(z)
        lg = torch.cat([self.sup(h), self.sub(h)], 1)
        return (lg, z, tok) if full else lg


def gate_kl(gate_logits, target):
    """KL(target || softmax(gate_logits)), mean over the batch. `target` rows are distributions with a positive floor (NB-03 epsilon)."""
    logq = F.log_softmax(gate_logits.float(), dim=-1)
    return (target * (torch.log(target.clamp_min(1e-8)) - logq)).sum(1).mean()

Writing cr_lib.py


In [4]:
sys.path.insert(0, WORK)
for nm, hint in (('eval_lib.py', 'cached-teacher-outputs-v1'), ('dean_lib.py', 'cached-teacher-outputs-v1'),
                 ('hod_lib.py', 'cached-teacher-outputs-v1'), ('cache_lib.py', 'cached-teacher-outputs-v1')):
    src = find_one(nm, hint)
    if os.path.abspath(src) != os.path.join(WORK, nm):
        shutil.copy(src, os.path.join(WORK, nm))
import eval_lib, dean_lib, hod_lib, cache_lib, cr_lib
for m in (eval_lib, dean_lib, hod_lib, cache_lib, cr_lib):
    importlib.reload(m)
ORIG_SCAN = dean_lib.selective_scan
dean_lib.selective_scan = hod_lib.fast_scan
from eval_lib import (sigmoid, auc_per_label, f1_at, tune_thresholds, summarize, bootstrap, paired_delta,
                      per_label_table, reliability, fit_calibration)
from cr_lib import ChiefResident, ChiefResidentDeploy, gate_kl

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
W2, W1 = 7.2, 3.5
mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'serif', 'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 7.5,
    'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6.5, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.4, 'legend.frameon': False, 'pdf.fonttype': 42, 'axes.linewidth': 0.6,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'lines.linewidth': 1.0,
})
SUPER = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
SUPER_COL = {'NORM': '#009E73', 'MI': '#D55E00', 'STTC': '#0072B2', 'CD': '#CC79A7', 'HYP': '#E69F00'}
FAMILY = {
    'AMI': 'MI', 'IMI': 'MI', 'LMI': 'MI', 'PMI': 'MI', 'ISCA': 'STTC', 'ISCI': 'STTC', 'ISC_': 'STTC', 'NST_': 'STTC',
    'CLBBB': 'CD', 'CRBBB': 'CD', 'ILBBB': 'CD', 'IRBBB': 'CD', 'IVCD': 'CD', 'LAFB/LPFB': 'CD', 'WPW': 'CD', '_AVB': 'CD',
    'LAO/LAE': 'HYP', 'LVH': 'HYP', 'RAO/RAE': 'HYP', 'RVH': 'HYP', 'SEHYP': 'HYP'}
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
C_CR, C_DEAN, C_HOD, C_PROF = '#0072B2', '#333333', '#56B4E9', '#D55E00'

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + '.png'), bbox_inches='tight', pad_inches=0.04)
    fig.savefig(os.path.join(FIG_DIR, name + '.pdf'), bbox_inches='tight', pad_inches=0.04)
    plt.show()

def panel(ax, s):
    ax.text(-0.02, 1.04, s, transform=ax.transAxes, fontsize=8.5, fontweight='bold', va='bottom', ha='right')
print('libraries ready | cuda mamba kernels:', dean_lib.USE_CUDA_MAMBA)

libraries ready | cuda mamba kernels: False


## 3. The cache, the labels and the signals
The cache is opened with its own reader. Every array of the Stage-7 manifest is checked for shape and dtype; the checks that matter for training (row order, label order, fold membership) are made against the CSV and `split_idx.npz`, not assumed. Signals are standardised with the statistics stored in `dean_finetuned.pt` (the same ones the teachers saw).

In [5]:
CHECKS = []
def check(name, ok, detail='', level='fail'):
    CHECKS.append(dict(check=name, status='PASS' if ok else ('FAIL' if level == 'fail' else 'WARN'), detail=str(detail)))
    print(f'[{"ok" if ok else level.upper()}] {name}' + (f': {detail}' if detail != '' else ''))
    return ok

man_path = find_one('teacher_cache_manifest.json', 'cached-teacher-outputs-v1')
CACHE_ROOT = os.path.dirname(man_path)
MAN = cache_lib.load_manifest(CACHE_ROOT)
problems = cache_lib.verify_cache(CACHE_ROOT, deep=False)
check('cache manifest: every file present with its recorded shape and dtype', not problems, problems[:3])
CK = {}
for k in ('y_train', 'y_val', 'ecg_id_train', 'ecg_id_val', 'fold_train', 'logit_offset', 'dean_logits_train_oof', 'dean_logits_train_insample',
          'dean_logits_val', 'hod_logits_train', 'hod_logits_val', 'prof_logits_train', 'prof_logits_val', 'prof_emb_train', 'prof_emb_val',
          'prof_reliable', 'route_train', 'route_val', 'dean_gram_train', 'dean_gram_val', 'dean_emb_val'):
    assert k in MAN['arrays'], f'{k} missing from the manifest: attach cached-teacher-outputs-v1 (Stage 7 version 3)'
    CK[k] = np.load(os.path.join(CACHE_ROOT, MAN['arrays'][k]['file']))

ck_t = torch.load(find_one('dean_finetuned.pt', 'dean-finetuned-v2'), map_location='cpu', weights_only=False)
LABELS = list(ck_t['label_columns']); ENC_CFG = dict(ck_t['enc_config']); ENC_CFG['grad_ckpt'] = False
MU, SD = np.array(ck_t['norm_mean'], np.float32), np.array(ck_t['norm_std'], np.float32)
D_TEACHER = ENC_CFG['d_model']
T_PARAMS = sum(p.numel() for p in ck_t['encoder'].values()) + sum(v.numel() for k, v in ck_t['head'].items() if k.startswith('fc.'))
del ck_t
assert LABELS[:5] == SUPER and len(LABELS) == 26
SUB = LABELS[5:]
check('manifest labels equal the teacher checkpoint labels', MAN['label_order'] == LABELS)

split = np.load(find_one('split_idx.npz', 'dean-finetuned-v2'), allow_pickle=True)
tr_idx, va_idx = split['train_idx'], split['val_idx']
df = pd.read_csv(find_one('labels_multihot.csv', 'ptbxl-processed-v1'))
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
fold_col = next(c for c in ('strat_fold', 'fold') if c in df.columns)
fold = df[fold_col].values.astype(int)
Yall = df[LABELS].values.astype(np.float32)
NTR, NVA = len(tr_idx), len(va_idx)
check('split: training = folds 1-8, validation = fold 9, fold 10 untouched',
      np.array_equal(tr_idx, np.where(fold <= 8)[0]) and np.array_equal(va_idx, np.where(fold == 9)[0]))
check('cache rows: labels equal the CSV labels in split order (train and validation)',
      np.array_equal(CK['y_train'].astype(np.float32), Yall[tr_idx]) and np.array_equal(CK['y_val'].astype(np.float32), Yall[va_idx]))
if 'ecg_id' in df.columns:
    check('cache rows: ecg ids equal the CSV ids in split order', np.array_equal(CK['ecg_id_train'], df['ecg_id'].values[tr_idx]) and np.array_equal(CK['ecg_id_val'], df['ecg_id'].values[va_idx]))
check('cache rows: fold_train matches', np.array_equal(CK['fold_train'], fold[tr_idx]))
for k, v in CK.items():
    if k not in ('logit_offset', 'prof_reliable'):
        check(f'{k}: all values finite', bool(np.isfinite(v.astype(np.float32, copy=False) if v.dtype != np.float16 else v.astype(np.float32)).all()), '', 'fail') if v.size < 6e7 else None

sig = np.load(find_one('ptbxl_100hz.npy', 'ptbxl-processed-v1'), mmap_mode='r')
if sig.ndim == 3 and sig.shape[1] != 12 and sig.shape[2] == 12:
    sig = np.transpose(sig, (0, 2, 1))
assert len(df) == sig.shape[0]
def load_std(idx):
    x = np.nan_to_num(np.ascontiguousarray(sig[idx]).astype(np.float32))
    return (x - MU[None, :, None]) / SD[None, :, None]
Y_tr, Y_va = Yall[tr_idx], Yall[va_idx]
REL = Y_va.sum(0) >= CFG['min_pos']
REL_SUB = REL[5:]
print(f'train {NTR} | val {NVA} | reliable labels {int(REL.sum())}/26, subclasses {int(REL_SUB.sum())}/21 | unreliable:', [LABELS[i] for i in np.where(~REL)[0]])
N_TOK = ENC_CFG['seq_len'] // ENC_CFG['patch']
P_DIM, N_PROF = CK['prof_emb_train'].shape[2], CK['prof_emb_train'].shape[1]
assert CK['dean_gram_train'].shape[1:] == (N_TOK, N_TOK) and CK['prof_emb_train'].shape[1] == 21
print(f'student tokens {N_TOK} | Professor embedding width {P_DIM} | teacher parameters {T_PARAMS / 1e6:.2f} M')

[ok] cache manifest: every file present with its recorded shape and dtype: []
[ok] manifest labels equal the teacher checkpoint labels
[ok] split: training = folds 1-8, validation = fold 9, fold 10 untouched
[ok] cache rows: labels equal the CSV labels in split order (train and validation)
[ok] cache rows: ecg ids equal the CSV ids in split order
[ok] cache rows: fold_train matches
[ok] y_train: all values finite
[ok] y_val: all values finite
[ok] ecg_id_train: all values finite
[ok] ecg_id_val: all values finite
[ok] fold_train: all values finite
[ok] dean_logits_train_oof: all values finite
[ok] dean_logits_train_insample: all values finite
[ok] dean_logits_val: all values finite
[ok] hod_logits_train: all values finite
[ok] hod_logits_val: all values finite
[ok] prof_logits_train: all values finite
[ok] prof_logits_val: all values finite
[ok] prof_emb_val: all values finite
[ok] route_train: all values finite
[ok] route_val: all values finite
[ok] dean_gram_train: all values finite


### 3.1 What the teachers know (a quick look, from the cache only)
The reference numbers every later comparison uses, on fold 9: DEAN (the generalist teacher), the HOD committee, the Professors (21 subclasses only). The in-sample versus out-of-fold gap on the training records is the reason the control arms of section 7 exist.

In [6]:
def mauc(Y, S, cols=None):
    cols = np.arange(Y.shape[1]) if cols is None else np.asarray(cols)
    a = auc_per_label(Y[:, cols], S[:, cols] if S.shape[1] == Y.shape[1] else S, 1)
    return float(np.nanmean(a[Y[:, cols].sum(0) >= CFG['min_pos']]))
REF = OrderedDict()
REF['DEAN'] = dict(val=CK['dean_logits_val'].astype(np.float32), cols=np.arange(26))
REF['HOD committee'] = dict(val=CK['hod_logits_val'].astype(np.float32), cols=np.arange(26))
REF['Professors'] = dict(val=CK['prof_logits_val'].astype(np.float32), cols=np.arange(5, 26))
TEACH = pd.DataFrame({n: dict(val_AUC=float(np.nanmean(auc_per_label(Y_va[:, r['cols']], r['val'], 1)[REL[r['cols']]])),
                              train_AUC_in_sample=float(np.nanmean(auc_per_label(Y_tr[:, r['cols']], {'DEAN': CK['dean_logits_train_insample'], 'HOD committee': CK['hod_logits_train'], 'Professors': CK['prof_logits_train']}[n].astype(np.float32), 1)[REL[r['cols']]])))
                      for n, r in REF.items()}).T
TEACH.loc['DEAN (out-of-fold)'] = [TEACH.loc['DEAN', 'val_AUC'], float(np.nanmean(auc_per_label(Y_tr, CK['dean_logits_train_oof'].astype(np.float32), 1)[REL]))]
TEACH['gap'] = TEACH.train_AUC_in_sample - TEACH.val_AUC
print(TEACH.round(4).to_string())
TEACH.to_csv(os.path.join(TAB_DIR, 'table_teachers_reference.csv'))

                    val_AUC  train_AUC_in_sample     gap
DEAN                 0.9283               0.9811  0.0528
HOD committee        0.9313               0.9828  0.0515
Professors           0.9338               0.9981  0.0643
DEAN (out-of-fold)   0.9283               0.9267 -0.0016


### 3.2 Moving everything to the GPU
The signals (0.8 GB), the Professor embeddings (0.19 GB), the Gram matrices and all logit targets fit on a T4 together; the training loop then never touches the CPU. Targets for every arm are built once as 26-column tensors in the student's label order.

In [7]:
PW_NP = np.exp(CK['logit_offset']).astype(np.float32)            # pos_weight of Stage 3 onwards: raw logits include log(pos_weight)
print('pos_weight range', PW_NP.min().round(2), PW_NP.max().round(2))
f32 = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32)).to(dev)
G = dict(
    X_tr=torch.from_numpy(load_std(tr_idx)).to(dev), X_va=torch.from_numpy(load_std(va_idx)).to(dev),
    Y_tr=f32(Y_tr), PW=f32(PW_NP),
    PE_tr=torch.from_numpy(CK['prof_emb_train']).to(dev), PE_va=torch.from_numpy(CK['prof_emb_val']).to(dev),     # fp16
    GR_tr=torch.from_numpy(CK['dean_gram_train']).to(dev),                                                     # fp16
    RT_tr=f32(CK['route_train']), RT_va=CK['route_val'].astype(np.float32),
)
prof_tr, prof_va = CK['prof_logits_train'].astype(np.float32), CK['prof_logits_val'].astype(np.float32)
hod_tr, dno_tr, dni_tr = (CK[k].astype(np.float32) for k in ('hod_logits_train', 'dean_logits_train_oof', 'dean_logits_train_insample'))
KD_TARGET = {                                                       # name -> (N,26) raw-logit target
    'chain':    np.concatenate([hod_tr[:, :5], prof_tr], 1),         # superclasses from the HODs, subclasses from the Professors
    'hod':      hod_tr,
    'dean_oof': dno_tr,
    'dean_in':  dni_tr,
    'ens':      np.concatenate([0.5 * (hod_tr[:, :5] + dno_tr[:, :5]), (prof_tr + hod_tr[:, 5:] + dno_tr[:, 5:]) / 3.0], 1),
}
KD_T = {k: f32(v) for k, v in KD_TARGET.items()}
# the same reference on the validation side (for the fidelity analyses)
KD_VAL = {'chain': np.concatenate([CK['hod_logits_val'][:, :5], prof_va], 1).astype(np.float32), 'dean_oof': CK['dean_logits_val'].astype(np.float32)}
print({k: tuple(v.shape) for k, v in KD_T.items()}, '| gpu memory', f'{torch.cuda.memory_allocated() / 1e9:.2f} GB' if USE_AMP else '(cpu)')
check('every arm target has 26 columns and the training row count', all(v.shape == (NTR, 26) for v in KD_T.values()))

pos_weight range 1.31 100.0
{'chain': (17418, 26), 'hod': (17418, 26), 'dean_oof': (17418, 26), 'dean_in': (17418, 26), 'ens': (17418, 26)} | gpu memory 1.22 GB
[ok] every arm target has 26 columns and the training row count


True

## 4. The student
The deployed network is `patchifier (Conv1d, 25 samples per token, 40 tokens) + positional embedding -> one bidirectional Mamba block (+ feed-forward) -> final LayerNorm -> mean over tokens -> head`. The head is a linear superclass layer and a two-layer subclass branch, exactly the layout of the HODs, so the student's 26 outputs sit in the same columns as every cached teacher output.

The width is the largest multiple of 16 (divisible by the number of Consultation heads) whose **deployed** parameter count stays under the budget. The table also lists the training-only parts that are discarded at the end.

In [8]:
def build_model(d_model, consult=True):
    return ChiefResident(ENC_CFG, d_model, 5, 21, CFG['head_drop'], CFG['sub_hidden'], 5,
                         P_DIM if consult else None, N_PROF, CFG['heads'], CFG['consult_drop']).to(dev)

def pick_width():
    if CFG['d_model']:
        return int(CFG['d_model'])
    best = None
    for d in range(64, 513, 16):
        if d % CFG['heads']:
            continue
        n = ChiefResident(ENC_CFG, d, 5, 21, CFG['head_drop'], CFG['sub_hidden'], 5, P_DIM, N_PROF, CFG['heads']).n_deployed()
        if n > CFG['param_budget']:
            break
        best = d
    assert best, 'no width fits the parameter budget'
    return best

D_CR = pick_width()
_m = build_model(D_CR)
PARAMS = dict(deployed=_m.n_deployed(), training_only=_m.n_training_only(),
              patchifier=sum(p.numel() for p in _m.enc.patch_embed.parameters()) + _m.enc.pos_emb.numel(),
              mamba_block=sum(p.numel() for p in _m.enc.blocks.parameters()),
              norm=sum(p.numel() for p in _m.enc.norm.parameters()),
              head=sum(p.numel() for n, p in _m.named_parameters() if n.startswith(('sup.', 'sub.'))),
              gate=sum(p.numel() for n, p in _m.named_parameters() if n.startswith('gate.')),
              consultation=sum(p.numel() for n, p in _m.named_parameters() if n.startswith('consult.')),
              aux_head=sum(p.numel() for n, p in _m.named_parameters() if n.startswith('aux_')))
assert PARAMS['patchifier'] + PARAMS['mamba_block'] + PARAMS['norm'] + PARAMS['head'] == PARAMS['deployed']
check(f'deployed network is under the parameter budget ({CFG["param_budget"] / 1e6:.2f} M)', PARAMS['deployed'] <= CFG['param_budget'], f'{PARAMS["deployed"] / 1e6:.3f} M at width {D_CR}')
print(pd.Series(PARAMS).to_frame('parameters').assign(share_of_teacher=lambda d: d.parameters / T_PARAMS).round(4).to_string())
print(f'\nwidth {D_CR} | deployed {PARAMS["deployed"] / 1e6:.3f} M = {100 * PARAMS["deployed"] / T_PARAMS:.1f} % of DEAN ({T_PARAMS / 1e6:.2f} M) | training-only {PARAMS["training_only"] / 1e6:.3f} M')
del _m

[ok] deployed network is under the parameter budget (1.00 M): 0.892 M at width 208
               parameters  share_of_teacher
deployed           892362            0.0777
training_only      434239            0.0378
patchifier          70928            0.0062
mamba_block        761072            0.0662
norm                  416            0.0000
head                59946            0.0052
gate                 1045            0.0001
consultation       373248            0.0325
aux_head            59946            0.0052

width 208 | deployed 0.892 M = 7.8 % of DEAN (11.49 M) | training-only 0.434 M


## 5. One training function for every arm
Each arm is a dictionary of switches (section 7). The loss of a step is

`w_sup·BCE(standalone) + w_kd·KD(standalone, target) + w_gate·KL(route ‖ gate) + w_rkd·MSE(Gram(student tokens), cached Gram) [+ w_aux·BCE(auxiliary) + w_sd·KD(standalone, auxiliary.detach())]`

and a term whose weight is zero is not computed. Notes on each term:

* **BCE** uses the pos-weights of Stage 3 (recovered from the cached `logit_offset`), so the student's raw logits live on the same scale as the cached teacher logits (which contain `log(pos_weight)`). KD in raw logit space is therefore consistent.
* **KD** is the sigmoid-KD of Stage 5 (temperature 3, `T²` times the KL, minimum zero).
* **Gate mimic** is a KL divergence to the cross-fitted routing distribution, from a linear head on the pooled representation.
* **RKD** compares the 40 × 40 token Gram matrix of the student with the cached DEAN Gram (centred tokens, cosine similarity), which needs no shared feature basis.
* **Auxiliary path** (arms with Consultation): the pooled representation attends over the cached Professor embeddings and feeds a separate head, trained with its own BCE; the standalone head is pulled towards it with a temperature-2 KD whose target is detached, so the standalone path *learns from* the auxiliary one but cannot change it.

Every epoch the EMA copy is validated on the standalone path only. The best EMA weights are kept. Checkpoints are written every `save_every` epochs and a finished run leaves `<tag>_final.pt`, which is reloaded instead of retrained if its configuration signature is unchanged.

In [9]:
ARM_DEFAULT = dict(w_kd=0.0, w_gate=0.0, w_rkd=0.0, consult=False, w_aux=0.0, w_sd=0.0, kd_src='chain')
def arm(**kw):
    a = dict(ARM_DEFAULT); a.update(kw); return a
THREE = dict(w_kd=CFG['w_kd'], w_gate=CFG['w_gate'], w_rkd=CFG['w_rkd'])
ARMS = OrderedDict([
    ('E_full',        arm(**THREE, consult=True, w_aux=CFG['w_aux'], w_sd=CFG['w_sd'])),     # the full proposal
    ('D_three_term',  arm(**THREE)),                                                          # corrected three-term loss, no Consultation
    ('A_sup',         arm()),                                                                 # supervised only
    ('B_kd',          arm(w_kd=CFG['w_kd'])),
    ('C_kd_gate',     arm(w_kd=CFG['w_kd'], w_gate=CFG['w_gate'])),
    ('K_consult_nosd', arm(**THREE, consult=True, w_aux=CFG['w_aux'], w_sd=0.0)),             # Consultation shapes the encoder but nothing is distilled from it
    ('F_dean_oof',    arm(**THREE, kd_src='dean_oof')),                                       # control: the only out-of-fold teacher
    ('G_hod',         arm(**THREE, kd_src='hod')),                                            # control: departmental level only
    ('H_dean_in',     arm(**THREE, kd_src='dean_in')),                                        # control: in-sample DEAN
    ('I_ensemble',    arm(**THREE, kd_src='ens')),                                            # control: mean of Professors, HODs, DEAN out-of-fold
])
SIG_KEYS = ['epochs', 'batch_size', 'wd', 'warmup_epochs', 'head_drop', 'sub_hidden', 'ema_decay', 'label_smooth', 'w_sup', 'T', 'T_sd',
            'aug_strength', 'prof_keep', 'consult_drop', 'heads', 'lr_floor']
def run_sig(arm_cfg, seed, lr, epochs):
    d = {k: CFG[k] for k in SIG_KEYS}; d.update(arm_cfg); d.update(seed=seed, lr=lr, epochs_run=epochs, width=D_CR, p_dim=P_DIM)
    return json.dumps(d, sort_keys=True, default=str)

@torch.no_grad()
def predict(net, X, prof=None, bs=256):
    """Returns dict of numpy arrays: logits, z, gate (and aux, att when the network has Consultation and `prof` is given).
    A batch with a non-finite output is recomputed in fp32 (the lesson of the Stage-6 supervised-only run)."""
    was = net.training
    net.eval()
    out = defaultdict(list)
    for i in range(0, len(X), bs):
        pb = prof[i:i + bs] if prof is not None else None
        with amp():
            o = net(X[i:i + bs], pb)
        if not all(torch.isfinite(v.float()).all() for k, v in o.items() if k != 'tok'):
            o = net(X[i:i + bs], pb)
        for k, v in o.items():
            if k != 'tok':
                out[k].append(v.float().cpu())
    net.train(was)
    return {k: torch.cat(v).numpy() for k, v in out.items()}

def macro_auc_rel(Y, S):
    a = auc_per_label(Y, S, 1)
    ok = Y.sum(0) >= CFG['min_pos']
    return float(np.nanmean(a[ok])) if ok.any() else float('nan')

def train_run(tag, arm_cfg, seed, lr, epochs, stop_after=None, quiet=False, resume=True):
    sig_now = run_sig(arm_cfg, seed, lr, epochs)
    fin_p, last_p = os.path.join(RUN_DIR, f'{tag}_final.pt'), os.path.join(RUN_DIR, f'{tag}_last.pt')
    consult = arm_cfg['consult']
    def make():
        return build_model(D_CR, consult)
    if resume and os.path.exists(fin_p):
        s = torch.load(fin_p, map_location='cpu', weights_only=False)
        if s['sig'] == sig_now:
            net = make(); net.load_state_dict(s['state']); net.eval()
            print(f'[{tag}] already trained (best epoch {s["best_epoch"]}, val AUC {s["best_score"]:.4f}), loaded')
            return dict(net=net, history=s['history'], best_epoch=s['best_epoch'], best_score=s['best_score'], tag=tag, arm=arm_cfg, seed=seed, lr=lr,
                        skipped=s.get('skipped', 0), finished=True)
        print(f'[{tag}] a finished run exists but its configuration differs, training again')
    set_seed(seed)
    net = make()
    net.set_prior(G['Y_tr'].mean(0).clamp(1e-4, 1 - 1e-4))
    ema = hod_lib.EMA(net, CFG['ema_decay'])
    dec = [p for n, p in net.named_parameters() if not (p.ndim < 2 or 'pos_emb' in n or 'prof_id' in n)]
    nod = [p for n, p in net.named_parameters() if (p.ndim < 2 or 'pos_emb' in n or 'prof_id' in n)]
    opt = torch.optim.AdamW([{'params': dec, 'weight_decay': CFG['wd']}, {'params': nod, 'weight_decay': 0.0}], lr=lr)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    X, Y, PW = G['X_tr'], G['Y_tr'], G['PW']
    TL = KD_T[arm_cfg['kd_src']]
    N, bs = len(X), CFG['batch_size']
    spe = N // bs
    total, warm = epochs * spe, CFG['warmup_epochs'] * spe
    start, step, history, best_score, best_epoch, best_state, bad, skipped = 0, 0, [], -1.0, 0, None, 0, 0
    if resume and os.path.exists(last_p):
        s = torch.load(last_p, map_location='cpu', weights_only=False)
        if s.get('sig') == sig_now:
            net.load_state_dict(s['model']); opt.load_state_dict(s['opt']); scaler.load_state_dict(s['scaler'])
            ema.model.load_state_dict(s['ema']); ema.n = s['ema_n']
            start, step, history = s['epoch'], s['step'], s['history']
            best_score, best_epoch, best_state, bad, skipped = s['best_score'], s['best_epoch'], s['best_state'], s['bad'], s['skipped']
            print(f'[{tag}] resumed at epoch {start} (best {best_score:.4f} at {best_epoch})')
    stopped_early = False
    sm = CFG['label_smooth']
    for epoch in range(start, epochs):
        net.train()
        t_ep = time.time()
        gen = torch.Generator().manual_seed(seed * 1000 + epoch)
        order = torch.randperm(N, generator=gen).to(dev)
        acc, n_ep_skip, consec = defaultdict(float), 0, 0
        bar = tqdm(range(spe), desc=f'[{tag}] epoch {epoch + 1}/{epochs}', leave=False, disable=quiet)
        for i in bar:
            idx = order[i * bs:(i + 1) * bs]
            x = hod_lib.aug_mild(X[idx], CFG['aug_strength']) if CFG['aug_strength'] > 0 else X[idx]
            y = Y[idx]
            sc = hod_lib.cosine_scale(step, total, warm, CFG['lr_floor'])
            for g in opt.param_groups:
                g['lr'] = lr * sc
            prof = keep = None
            if consult:
                prof = G['PE_tr'][idx]
                keep = torch.rand(len(idx), N_PROF, device=dev) < CFG['prof_keep']
                keep[torch.arange(len(idx), device=dev), torch.randint(0, N_PROF, (len(idx),), device=dev)] = True
            with amp():
                o = net(x, prof, keep)
            lg = o['logits'].float()
            tgt = y * (1 - sm) + 0.5 * sm
            l_sup = F.binary_cross_entropy_with_logits(lg, tgt, pos_weight=PW)
            loss = CFG['w_sup'] * l_sup
            l_kd = l_gate = l_rkd = l_aux = l_sd = torch.zeros((), device=dev)
            if arm_cfg['w_kd'] > 0:
                l_kd = hod_lib.kd_sigmoid(lg, TL[idx], CFG['T']); loss = loss + arm_cfg['w_kd'] * l_kd
            if arm_cfg['w_gate'] > 0:
                l_gate = gate_kl(o['gate'], G['RT_tr'][idx]); loss = loss + arm_cfg['w_gate'] * l_gate
            if arm_cfg['w_rkd'] > 0:
                l_rkd = F.mse_loss(hod_lib.token_gram(o['tok']), G['GR_tr'][idx].float()); loss = loss + arm_cfg['w_rkd'] * l_rkd
            if consult and arm_cfg['w_aux'] > 0:
                aux = o['aux'].float()
                l_aux = F.binary_cross_entropy_with_logits(aux, tgt, pos_weight=PW); loss = loss + arm_cfg['w_aux'] * l_aux
                if arm_cfg['w_sd'] > 0:
                    l_sd = hod_lib.kd_sigmoid(lg, aux.detach(), CFG['T_sd']); loss = loss + arm_cfg['w_sd'] * l_sd
            if not torch.isfinite(loss):
                skipped += 1; n_ep_skip += 1; consec += 1
                opt.zero_grad(set_to_none=True)
                if consec > CFG['max_skip']:
                    raise RuntimeError(f'[{tag}] {consec} non-finite losses in a row at step {step}: lower the learning rate (now {lr:g})')
                continue
            consec = 0
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), CFG['clip'])
            scaler.step(opt); scaler.update()
            ema.update(net)
            step += 1
            for k, v in (('loss', loss), ('sup', l_sup), ('kd', l_kd), ('gate', l_gate), ('rkd', l_rkd), ('aux', l_aux), ('sd', l_sd)):
                acc[k] += float(v)
            if i % 20 == 0:
                bar.set_postfix(loss=f'{acc["loss"] / (i + 1 - n_ep_skip):.3f}')
        bar.close()
        if USE_AMP: torch.cuda.synchronize()
        ep_time = time.time() - t_ep
        pv = predict(ema.model, G['X_va'], G['PE_va'] if consult else None)
        score = macro_auc_rel(Y_va, pv['logits'])
        score = -1.0 if not np.isfinite(score) else score
        done = epoch + 1
        row = {'epoch': done, 'lr': lr * sc, 'sec_per_epoch': ep_time, 'skipped': n_ep_skip, 'val_macro_auc': score,
               'val_logit_mae_chain': float(np.abs(pv['logits'] - KD_VAL['chain']).mean()),
               'val_gate_kl': float((G['RT_va'] * (np.log(G['RT_va']) - torch.log_softmax(torch.from_numpy(pv['gate']), -1).numpy())).sum(1).mean())}
        if 'aux' in pv:
            row['val_aux_auc'] = macro_auc_rel(Y_va, pv['aux'])
        row.update({k: v / max(spe - n_ep_skip, 1) for k, v in acc.items()})
        history.append(row)
        improved = score > best_score
        if improved:
            best_score, best_epoch, bad = score, done, 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        else:
            bad += 1
        stop_now = bool(CFG['max_hours']) and done < epochs and 1.5 * ep_time / 3600 > hours_left(CFG['max_hours'])
        cut = stop_after is not None and done >= stop_after
        if done % CFG['save_every'] == 0 or done == epochs or stop_now or bad >= CFG['patience'] or cut:
            torch.save({'model': net.state_dict(), 'ema': ema.model.state_dict(), 'ema_n': ema.n, 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                        'epoch': done, 'step': step, 'history': history, 'best_score': best_score, 'best_epoch': best_epoch, 'best_state': best_state,
                        'bad': bad, 'skipped': skipped, 'sig': sig_now}, last_p + '.tmp')
            os.replace(last_p + '.tmp', last_p)
        if not quiet:
            print(f"[{tag}] ep {done:2d} | loss {row['loss']:.3f} (sup {row['sup']:.3f} kd {row['kd']:.3f} gate {row['gate']:.3f} rkd {row['rkd']:.4f} aux {row['aux']:.3f} sd {row['sd']:.3f}) | "
                  f"val AUC {score:.4f}" + (f" (aux {row['val_aux_auc']:.4f})" if 'val_aux_auc' in row else '') + f" | {ep_time:4.0f}s | skipped {n_ep_skip}{' | best' if improved else ''}")
        if bad >= CFG['patience']:
            print(f'[{tag}] no improvement for {CFG["patience"]} epochs, stopping at epoch {done}'); break
        if stop_now:
            stopped_early = True
            print(f'[{tag}] time budget reached after epoch {done}; state saved, publish the output as a Dataset and re-run to resume'); break
        if cut:
            stopped_early = True; break
    if stopped_early:
        return dict(net=net, history=history, best_epoch=best_epoch, best_score=best_score, tag=tag, arm=arm_cfg, seed=seed, lr=lr, skipped=skipped, finished=False)
    net.load_state_dict(best_state)
    net.eval()
    torch.save({'state': net.state_dict(), 'history': history, 'best_epoch': best_epoch, 'best_score': best_score, 'sig': sig_now, 'skipped': skipped,
                'arm': arm_cfg, 'seed': seed, 'lr': lr, 'width': D_CR}, fin_p + '.tmp')
    os.replace(fin_p + '.tmp', fin_p)
    return dict(net=net, history=history, best_epoch=best_epoch, best_score=best_score, tag=tag, arm=arm_cfg, seed=seed, lr=lr, skipped=skipped, finished=True)

## 6. Smoke test
Real code on a few real batches, before anything long starts. It checks that

* the fast scan reproduces the library scan (outputs and gradients);
* each loss behaves on constructed inputs: KD is zero when student equals target, the gate KL is zero when the gate equals its target and positive otherwise, the Gram term is rotation-invariant;
* **Consultation** attention weights sum to one, a dropped Professor gets no weight, and the module reacts to the Professors it is given;
* **the standalone path does not depend on the Professors** (the logits are bit-identical when the cached Professor embeddings are replaced by noise), and the deployed class holds no training-only parameter;
* every parameter of the full model receives a finite gradient from the combined loss, and the gate and Consultation parameters receive none from a supervised-only loss;
* two epochs run, the final checkpoint round-trips, a stopped run resumes, and the time per step is measured so the run times below can be projected.

In [10]:
def smoke_test():
    print('=' * 70, '\nSMOKE TEST\n' + '=' * 70)
    torch.manual_seed(0)
    B, L, Din, Ns = 4, 40, 16, 8
    u, dt = torch.randn(B, L, Din, device=dev, requires_grad=True), (torch.rand(B, L, Din, device=dev) * 0.5).requires_grad_()
    A = (-(torch.rand(Din, Ns, device=dev) + 1)).requires_grad_()
    Bm, Cm = torch.randn(B, L, Ns, device=dev, requires_grad=True), torch.randn(B, L, Ns, device=dev, requires_grad=True)
    Dp = torch.ones(Din, device=dev)
    o1 = ORIG_SCAN(u, dt, A, Bm, Cm, Dp); g1 = torch.autograd.grad(o1.sum(), [u, dt, A, Bm, Cm])
    o2 = hod_lib.fast_scan(u, dt, A, Bm, Cm, Dp); g2 = torch.autograd.grad(o2.sum(), [u, dt, A, Bm, Cm])
    assert torch.allclose(o1, o2, atol=1e-3, rtol=1e-3) and all(torch.allclose(a, b, atol=1e-2, rtol=1e-2) for a, b in zip(g1, g2))
    print('[ok] fast scan matches the library scan')
    t = torch.randn(8, 6, device=dev) * 2
    assert float(hod_lib.kd_sigmoid(t, t, 3.0)) < 1e-4 and float(hod_lib.kd_sigmoid(t + 1.5, t, 3.0)) > 0.01
    tg = torch.softmax(torch.randn(8, 5, device=dev), -1)
    assert float(gate_kl(torch.log(tg), tg)) < 1e-6 and float(gate_kl(torch.randn(8, 5, device=dev), tg)) > 1e-3
    tok = torch.randn(4, 40, 32, device=dev); Q, _ = torch.linalg.qr(torch.randn(32, 32, device=dev))
    assert float(hod_lib.gram_loss(tok @ Q, tok)) < 1e-8
    print('[ok] KD is 0 for equal logits, gate KL is 0 for equal distributions, Gram is rotation-invariant')
    # model
    net = build_model(D_CR, True); net.train()
    idx = torch.arange(CFG['batch_size'], device=dev) % NTR
    x, y = hod_lib.aug_mild(G['X_tr'][idx]), G['Y_tr'][idx]
    prof = G['PE_tr'][idx]
    keep = torch.rand(len(idx), N_PROF, device=dev) < 0.5
    keep[:, 0] = True
    # Consultation behaviour
    z = torch.randn(len(idx), D_CR, device=dev)
    zc, att = net.consult(z, prof, keep)
    assert torch.allclose(att.sum(-1), torch.ones_like(att.sum(-1)), atol=1e-4), 'attention rows must sum to one'
    assert float(att[:, :, :][~keep[:, None, :].expand_as(att)].max()) < 1e-6, 'dropped Professors must get no weight'
    net.eval()
    zc2, _ = net.consult(z, torch.randn_like(prof), None); zc1, _ = net.consult(z, prof, None)
    assert float((zc1 - zc2).abs().mean()) > 1e-4, 'Consultation ignores the Professor embeddings'
    print('[ok] Consultation: attention sums to one, dropped Professors get no weight, the output depends on the embeddings')
    # standalone path is blind to the Professors
    net.eval()
    with torch.no_grad():
        a = net(x, prof)['logits']; b = net(x, torch.randn_like(prof))['logits']; c = net(x, None)['logits']
    assert torch.equal(a, b) and torch.equal(a, c), 'the standalone path depends on the Professor embeddings'
    dep = ChiefResidentDeploy(ENC_CFG, D_CR, 5, 21, CFG['head_drop'], CFG['sub_hidden']).to(dev).eval()
    dep.load_state_dict(net.deploy_state(), strict=True)
    assert not any(n.startswith(cr_lib.TRAIN_ONLY_PREFIXES) for n, _ in dep.named_parameters())
    with torch.no_grad():
        assert torch.allclose(dep(x), a, atol=1e-5)
    print(f'[ok] standalone logits are identical with real, random or no Professor embeddings; the deployed class has {sum(p.numel() for p in dep.parameters()) / 1e6:.3f} M parameters and no training-only module')
    # gradients
    net.train()
    out = net(x, prof, keep)
    lg = out['logits'].float()
    pw = G['PW']
    loss = (F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw) + hod_lib.kd_sigmoid(lg, KD_T['chain'][idx], 3.0) + gate_kl(out['gate'], G['RT_tr'][idx])
            + F.mse_loss(hod_lib.token_gram(out['tok']), G['GR_tr'][idx].float()) + F.binary_cross_entropy_with_logits(out['aux'].float(), y, pos_weight=pw)
            + hod_lib.kd_sigmoid(lg, out['aux'].float().detach(), 2.0))
    loss.backward()
    bad = [n for n, p in net.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
    zero = [n for n, p in net.named_parameters() if p.grad is not None and float(p.grad.abs().sum()) == 0.0 and 'bias' not in n]
    assert not bad, bad[:6]
    print(f'[ok] every parameter of the full model receives a finite gradient ({sum(p.numel() for p in net.parameters()) / 1e6:.3f} M in total; all-zero weight gradients: {zero[:3]})')
    net.zero_grad(set_to_none=True)
    out = net(x, prof, keep)
    F.binary_cross_entropy_with_logits(out['logits'].float(), y, pos_weight=pw).backward()
    leak = [n for n, p in net.named_parameters() if n.startswith(cr_lib.TRAIN_ONLY_PREFIXES) and p.grad is not None and float(p.grad.abs().sum()) > 0]
    assert not leak, leak[:5]
    print('[ok] the supervised loss of the standalone path sends no gradient into the gate, Consultation or auxiliary head')
    # time
    net.train()
    opt = torch.optim.AdamW(net.parameters(), lr=1e-4); scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    ts = []
    for k in range(6):
        if USE_AMP: torch.cuda.synchronize()
        t0 = time.time()
        with amp():
            o = net(hod_lib.aug_mild(G['X_tr'][idx]), prof, keep)
        l = F.binary_cross_entropy_with_logits(o['logits'].float(), y, pos_weight=pw) + F.binary_cross_entropy_with_logits(o['aux'].float(), y, pos_weight=pw)
        opt.zero_grad(); scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        if USE_AMP: torch.cuda.synchronize()
        ts.append(time.time() - t0)
    sps = float(np.median(ts[2:]))
    spe = NTR // CFG['batch_size']
    print(f'[ok] {sps:.3f} s per step at batch {CFG["batch_size"]} -> about {sps * spe:.0f} s per epoch')
    return sps * spe

EPOCH_SEC = smoke_test()
n_runs = len(ARMS) + len([a for a in CFG['seed_arms']]) * (len(CFG['seeds']) - 1)
print(f'\nprojected: {n_runs} full runs x up to {CFG["epochs"]} epochs x {EPOCH_SEC:.0f} s = {n_runs * CFG["epochs"] * EPOCH_SEC / 3600:.1f} h (early stopping usually cuts this by a third), pilot {len(CFG["pilot_lrs"]) * CFG["pilot_epochs"] * EPOCH_SEC / 60:.0f} min')

SMOKE TEST
[ok] fast scan matches the library scan
[ok] KD is 0 for equal logits, gate KL is 0 for equal distributions, Gram is rotation-invariant
[ok] Consultation: attention sums to one, dropped Professors get no weight, the output depends on the embeddings
[ok] standalone logits are identical with real, random or no Professor embeddings; the deployed class has 0.892 M parameters and no training-only module
[ok] every parameter of the full model receives a finite gradient (1.327 M in total; all-zero weight gradients: [])
[ok] the supervised loss of the standalone path sends no gradient into the gate, Consultation or auxiliary head
[ok] 0.065 s per step at batch 64 -> about 18 s per epoch

projected: 16 full runs x up to 40 epochs x 18 s = 3.2 h (early stopping usually cuts this by a third), pilot 7 min


## 7. The arms
Ten arms, each answering one question. All share the student, the data, the optimiser and the number of epochs; they differ only in the switches below.

| Arm | KD target | Gate | RKD | Consultation + self-distillation | Question it answers |
|---|---|---|---|---|---|
| `A_sup` | none | no | no | no | what the architecture reaches with labels alone |
| `B_kd` | chain | no | no | no | does logit distillation from the cascade help? |
| `C_kd_gate` | chain | yes | no | no | does matching the gate help on top? |
| `D_three_term` | chain | yes | yes | no | the corrected three-term loss of the synopsis |
| `E_full` | chain | yes | yes | yes | **the full proposal** (three terms + Consultation + self-distillation) |
| `K_consult_nosd` | chain | yes | yes | auxiliary head only | does Consultation help the encoder even if nothing is distilled from it? |
| `F_dean_oof` | DEAN out-of-fold, all 26 | yes | yes | no | control: the only honest teacher, no hierarchy |
| `G_hod` | HOD committee, all 26 | yes | yes | no | control: departmental tier only |
| `H_dean_in` | DEAN in-sample, all 26 | yes | yes | no | control: the same teacher with the in-sample sharpness problem |
| `I_ensemble` | mean of Professors, HODs and DEAN out-of-fold | yes | yes | no | control: do the tiers add up? |

`chain` = HOD committee for the 5 superclass columns, Professors for the 21 subclass columns. The hierarchy claim of the project is supported only if `D_three_term` (or `E_full`) beats `F_dean_oof` and `G_hod`, whose paired confidence interval excludes zero; it is **not** supported by `D` beating `A`, which only shows that distillation from anything helps. Three arms (`A`, `D`, `E`) are trained with three seeds so that the differences between them can be compared with seed noise.

## 8. Learning-rate pilot
The full model (`E_full`) is trained for `pilot_epochs` at each candidate rate with one seed. A rate is **eligible** only if its run skipped no batch. Among eligible rates the lowest is kept unless a higher one has a validation macro-AUC better by more than `pilot_margin`. The rule is fixed here, before the numbers are seen.

In [11]:
RES = OrderedDict()
PILOT = []
if CFG['run_lr_pilot'] and CFG['lr'] is None:
    for lr in CFG['pilot_lrs']:
        r = train_run(f'pilot_lr{lr:g}', ARMS['E_full'], 0, lr, CFG['pilot_epochs'])
        PILOT.append(dict(lr=lr, val_macro_auc=r['best_score'], best_epoch=r['best_epoch'], skipped=r['skipped'], last_loss=r['history'][-1]['loss'],
                          sec_per_epoch=float(np.mean([h['sec_per_epoch'] for h in r['history']])), eligible=r['skipped'] == 0))
        del r
        if USE_AMP: torch.cuda.empty_cache()
    PILOT = pd.DataFrame(PILOT)
    el = PILOT[PILOT.eligible].sort_values('lr')
    if len(el):
        pick_ = el.iloc[0]
        for _, row in el.iloc[1:].iterrows():
            if row.val_macro_auc > pick_.val_macro_auc + CFG['pilot_margin']:
                pick_ = row
        CFG['lr'] = float(pick_.lr)
    else:
        CFG['lr'] = float(min(CFG['pilot_lrs'])) / 2
        print('WARNING: every pilot rate skipped batches; falling back to half the smallest rate')
    PILOT.to_csv(os.path.join(TAB_DIR, 'lr_pilot.csv'), index=False)
    print(PILOT.round(4).to_string(index=False), f'\nchosen learning rate: {CFG["lr"]:g}')
else:
    CFG['lr'] = CFG['lr'] or CFG['lr_default']
    print('learning-rate pilot not run; using', CFG['lr'])
LR = CFG['lr']

[pilot_lr0.0003] epoch 1/8:   0%|          | 0/272 [00:00<?, ?it/s]

[pilot_lr0.0003] ep  1 | loss 6.070 (sup 2.475 kd 0.931 gate 0.614 rkd 0.0851 aux 2.146 sd 0.169) | val AUC 0.7353 (aux 0.8032) |   18s | skipped 0 | best


[pilot_lr0.0003] epoch 2/8:   0%|          | 0/272 [00:00<?, ?it/s]

[pilot_lr0.0003] ep  2 | loss 2.762 (sup 1.147 kd 0.301 gate 0.319 rkd 0.0744 aux 0.963 sd 0.153) | val AUC 0.8511 (aux 0.8724) |   18s | skipped 0 | best


[pilot_lr0.0003] epoch 3/8:   0%|          | 0/272 [00:00<?, ?it/s]

[pilot_lr0.0003] ep  3 | loss 2.346 (sup 1.033 kd 0.197 gate 0.224 rkd 0.0623 aux 0.769 sd 0.204) | val AUC 0.8719 (aux 0.8695) |   18s | skipped 0 | best


... (38 outputs left out)

[pilot_lr0.001] epoch 7/8:   0%|          | 0/272 [00:00<?, ?it/s]

[pilot_lr0.001] ep  7 | loss 1.653 (sup 0.773 kd 0.073 gate 0.080 rkd 0.0486 aux 0.579 sd 0.164) | val AUC 0.9121 (aux 0.8786) |   18s | skipped 0


[pilot_lr0.001] epoch 8/8:   0%|          | 0/272 [00:00<?, ?it/s]

[pilot_lr0.001] ep  8 | loss 1.611 (sup 0.752 kd 0.069 gate 0.073 rkd 0.0485 aux 0.574 sd 0.156) | val AUC 0.9112 (aux 0.8741) |   18s | skipped 0
    lr  val_macro_auc  best_epoch  skipped  last_loss  sec_per_epoch  eligible
0.0003         0.8971           8        0     1.8709        17.8553      True
0.0006         0.9105           8        0     1.6983        17.7052      True
0.0010         0.9133           6        0     1.6112        17.7438      True 
chosen learning rate: 0.001


## 9. Training the arms
The main model is trained first, so that if the session ends early the most important result exists. A run is skipped (and said so) only when the remaining time budget cannot hold it. Every finished run leaves `runs/<tag>_final.pt`; re-running the notebook reloads it instead of training again.

In [12]:
RUN_ORDER = [(a, 0) for a in ARMS] + [(a, s) for s in CFG['seeds'][1:] for a in CFG['seed_arms']]
PRED = {}
def collect(tag, r):
    net = r['net']; cons = r['arm']['consult']
    pv = predict(net, G['X_va'], G['PE_va'] if cons else None)
    pt = predict(net, G['X_tr'], G['PE_tr'] if cons else None)
    keep_keys = ('logits', 'gate', 'aux', 'att', 'z')
    PRED[tag] = dict(val={k: v for k, v in pv.items() if k in keep_keys}, train={k: v for k, v in pt.items() if k in ('logits', 'aux', 'z') and (k != 'z' or tag in ('E_full_s0', 'D_three_term_s0'))})

TIMES = []
for a, s in RUN_ORDER:
    tag = f'{a}_s{s}'
    est = CFG['epochs'] * (np.mean(TIMES) if TIMES else EPOCH_SEC) / 3600
    if CFG['max_hours'] and tag != 'E_full_s0' and hours_left(CFG['max_hours']) < 1.1 * est * 0.6:
        print(f'[{tag}] skipped: {hours_left(CFG["max_hours"]):.1f} h left, a run needs up to {est:.1f} h'); continue
    t0 = time.time()
    r = train_run(tag, ARMS[a], s, LR, CFG['epochs'])
    if not r['finished']:
        print('stopping here: state saved'); RES[tag] = r; break
    RES[tag] = r
    collect(tag, r)
    TIMES.append(float(np.mean([h['sec_per_epoch'] for h in r['history']])))
    print(f'[{tag}] done in {(time.time() - t0) / 60:.1f} min | best epoch {r["best_epoch"]} | val macro-AUC {r["best_score"]:.4f} | skipped batches {r["skipped"]}\n')
    if tag not in ('E_full_s0', 'D_three_term_s0'):
        r['net'] = r['net'].cpu()
    if USE_AMP: torch.cuda.empty_cache()
assert 'E_full_s0' in PRED, 'the main model was not trained'
SUMMARY = pd.DataFrame([dict(tag=t, arm=r['arm'].get('name', t.rsplit('_s', 1)[0]), seed=r['seed'], best_epoch=r['best_epoch'], epochs_run=len(r['history']),
                              val_macro_auc=r['best_score'], skipped=r['skipped'], sec_per_epoch=float(np.mean([h['sec_per_epoch'] for h in r['history']])))
                         for t, r in RES.items()])
print(SUMMARY.round(4).to_string(index=False))
SUMMARY.to_csv(os.path.join(TAB_DIR, 'runs_summary.csv'), index=False)
check('no batch was skipped in any run', int(SUMMARY.skipped.sum()) == 0, f'{int(SUMMARY.skipped.sum())} skipped', level='warn')
MAIN = 'E_full_s0'
check('main model: standalone validation macro-AUC is far above chance (rows and signals are aligned)', RES[MAIN]['best_score'] > 0.8, f'{RES[MAIN]["best_score"]:.4f}')

[E_full_s0] epoch 1/40:   0%|          | 0/272 [00:00<?, ?it/s]

[E_full_s0] ep  1 | loss 4.718 (sup 1.916 kd 0.676 gate 0.501 rkd 0.0819 aux 1.666 sd 0.168) | val AUC 0.8372 (aux 0.8657) |   18s | skipped 0 | best


[E_full_s0] epoch 2/40:   0%|          | 0/272 [00:00<?, ?it/s]

[E_full_s0] ep  2 | loss 2.440 (sup 1.064 kd 0.211 gate 0.249 rkd 0.0646 aux 0.786 sd 0.223) | val AUC 0.8740 (aux 0.8735) |   18s | skipped 0 | best


[E_full_s0] epoch 3/40:   0%|          | 0/272 [00:00<?, ?it/s]

[E_full_s0] ep  3 | loss 2.176 (sup 0.980 kd 0.152 gate 0.187 rkd 0.0572 aux 0.699 sd 0.222) | val AUC 0.8956 (aux 0.8774) |   18s | skipped 0 | best


... (68 outputs left out)

[A_sup_s0] epoch 4/40:   0%|          | 0/272 [00:00<?, ?it/s]

[A_sup_s0] ep  4 | loss 0.962 (sup 0.962 kd 0.000 gate 0.000 rkd 0.0000 aux 0.000 sd 0.000) | val AUC 0.8808 |   12s | skipped 82 | best


[A_sup_s0] epoch 5/40:   0%|          | 0/272 [00:00<?, ?it/s]

RuntimeError: [A_sup_s0] 26 non-finite losses in a row at step 998: lower the learning rate (now 0.001)

## 10. Evaluation on fold 9
Everything below uses the validation fold, which was also used to select the epoch and the learning rate (the same situation as every earlier stage). The test fold stays unread until Stage 10.

**Statistics.** One set of 300 bootstrap resamples of the validation records is drawn once and every model is scored on the *same* resamples, so any difference between two models is a paired difference (the resampling noise cancels). A labelled macro-AUC averages only the labels with at least 10 validation positives. A claim is called **supported** if the 95 % interval of the paired difference excludes zero in the claimed direction, **contradicted** if it excludes zero in the other direction, and **not supported** if it includes zero. Nothing is claimed from a point estimate alone.

In [ ]:
from scipy.stats import rankdata
LOG, AVAIL = OrderedDict(), {}
def add_model(name, Z, cols=None):
    full = np.zeros((NVA, 26), np.float32)
    av = np.zeros(26, bool)
    cols = np.arange(26) if cols is None else np.asarray(cols)
    full[:, cols] = Z; av[cols] = True
    LOG[name] = full; AVAIL[name] = av
add_model('DEAN', REF['DEAN']['val']); add_model('HOD committee', REF['HOD committee']['val'])
add_model('Professors', REF['Professors']['val'], REF['Professors']['cols'])
for tag in PRED:
    add_model(tag, PRED[tag]['val']['logits'])
CR = MAIN
rng_b = np.random.default_rng(CFG['seed'])
BIDX = rng_b.integers(0, NVA, (CFG['n_boot'], NVA))

def auc_cols(Y, S):
    npos = Y.sum(0); nneg = len(Y) - npos
    r = rankdata(S, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        a = ((r * Y).sum(0) - npos * (npos + 1) / 2) / (npos * nneg)
    a[(npos == 0) | (nneg == 0)] = np.nan
    return a
assert np.allclose(np.nan_to_num(auc_cols(Y_va, LOG['DEAN'])[REL]), np.nan_to_num(auc_per_label(Y_va, LOG['DEAN'], 1)[REL]), atol=1e-9), 'fast AUC disagrees with eval_lib'
POINT = {n: auc_cols(Y_va, S) for n, S in LOG.items()}
BOOT = {n: np.stack([auc_cols(Y_va[ix], S[ix]) for ix in BIDX]) for n, S in LOG.items()}
SCOPE = dict(all=np.ones(26, bool), super=np.arange(26) < 5, sub=np.arange(26) >= 5)
def mask_for(names, scope='all'):
    m = REL & SCOPE[scope]
    for n in names:
        m = m & AVAIL[n]
    return m
def macro(name, scope='all', boot=False):
    m = mask_for([name], scope)
    return np.nanmean(BOOT[name][:, m], 1) if boot else float(np.nanmean(POINT[name][m]))
def ci(v):
    return [float(np.mean(v)), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
def delta(a, b, scope='all'):
    m = mask_for([a, b], scope)
    d = np.nanmean(BOOT[a][:, m], 1) - np.nanmean(BOOT[b][:, m], 1)
    return dict(delta=float(np.nanmean(POINT[a][m]) - np.nanmean(POINT[b][m])), lo=float(np.percentile(d, 2.5)), hi=float(np.percentile(d, 97.5)), n_labels=int(m.sum()))
def verdict(d, margin=0.0):
    if d['lo'] > margin: return 'supported'
    if d['hi'] < -margin: return 'contradicted'
    return 'not supported'

rows = []
for n in LOG:
    for sc in ('all', 'super', 'sub'):
        if mask_for([n], sc).any():
            v = macro(n, sc, True)
            rows.append(dict(model=n, scope=sc, macro_auc=macro(n, sc), lo=float(np.percentile(v, 2.5)), hi=float(np.percentile(v, 97.5)), n_labels=int(mask_for([n], sc).sum())))
MAIN_TAB = pd.DataFrame(rows)
MAIN_TAB.to_csv(os.path.join(TAB_DIR, 'table_all_models_auc.csv'), index=False)
show = MAIN_TAB[MAIN_TAB.model.isin(['DEAN', 'HOD committee', 'Professors', CR])].pivot(index='model', columns='scope', values='macro_auc').round(4)
print(show.to_string())

### 10.1 The claims table
Every claim of the project that this stage can test, with the paired difference of macro-AUC (fold 9, labels with at least 10 positives), its 95 % interval, and the verdict by the rule above. Reading it: *supported* is a result, *not supported* means the data cannot tell, *contradicted* is a result against the claim and is reported as such.

In [ ]:
def S0(a): return f'{a}_s0'
CLAIMS_DEF = [
    ('C1', 'logit distillation from the cascade improves the student', S0('B_kd'), S0('A_sup'), 'all', 0.0),
    ('C2', 'matching the gate adds to distillation', S0('C_kd_gate'), S0('B_kd'), 'all', 0.0),
    ('C3', 'Gram-RKD adds to distillation and gate', S0('D_three_term'), S0('C_kd_gate'), 'all', 0.0),
    ('C4', 'the three-term loss beats supervised training', S0('D_three_term'), S0('A_sup'), 'all', 0.0),
    ('C5', 'Consultation + self-distillation improves the standalone path', S0('E_full'), S0('D_three_term'), 'all', 0.0),
    ('C6', 'self-distillation matters (full model against auxiliary head without distillation)', S0('E_full'), S0('K_consult_nosd'), 'all', 0.0),
    ('C7', 'the auxiliary head alone shapes a better encoder', S0('K_consult_nosd'), S0('D_three_term'), 'all', 0.0),
    ('C8', 'HIERARCHY: the cascade target beats DEAN out-of-fold as the only teacher', S0('D_three_term'), S0('F_dean_oof'), 'all', 0.0),
    ('C9', 'HIERARCHY: the cascade target beats the departmental tier alone', S0('D_three_term'), S0('G_hod'), 'all', 0.0),
    ('C10', 'the cascade target beats in-sample DEAN logits', S0('D_three_term'), S0('H_dean_in'), 'all', 0.0),
    ('C11', 'the cascade target beats the equal-weight ensemble of all tiers', S0('D_three_term'), S0('I_ensemble'), 'all', 0.0),
    ('C12', 'HIERARCHY (full model): proposal beats DEAN out-of-fold as the only teacher', S0('E_full'), S0('F_dean_oof'), 'all', 0.0),
    ('C13', 'the Chief Resident is not worse than DEAN by more than 0.01 AUC (non-inferiority)', CR, 'DEAN', 'all', -0.01),
    ('C14', 'the Chief Resident is not worse than the HOD committee by more than 0.01 AUC (non-inferiority)', CR, 'HOD committee', 'all', -0.01),
    ('C15', 'the Chief Resident is not worse than the Professors by more than 0.01 AUC on subclasses (non-inferiority)', CR, 'Professors', 'sub', -0.01),
    ('C16', 'the Chief Resident is better than DEAN', CR, 'DEAN', 'all', 0.0),
]
CL = []
for cid, text, a, b, sc, margin in CLAIMS_DEF:
    if a not in LOG or b not in LOG:
        continue
    d = delta(a, b, sc)
    CL.append(dict(id=cid, claim=text, a=a, b=b, scope=sc, margin=margin, **d, verdict=verdict(d, margin)))
CLAIMS = pd.DataFrame(CL)
CLAIMS.to_csv(os.path.join(TAB_DIR, 'table_claims.csv'), index=False)
with pd.option_context('display.max_colwidth', 90, 'display.width', 250):
    print(CLAIMS[['id', 'claim', 'delta', 'lo', 'hi', 'verdict']].round(4).to_string(index=False))

### 10.2 Seeds
`A`, `D` and `E` were trained with three seeds. The seed-to-seed spread is the yardstick for differences between arms that only have one seed.

In [ ]:
SEEDTAB = []
for a in CFG['seed_arms']:
    tags = [f'{a}_s{s}' for s in CFG['seeds'] if f'{a}_s{s}' in LOG]
    v = [macro(t) for t in tags]
    SEEDTAB.append(dict(arm=a, n_seeds=len(tags), mean=float(np.mean(v)), sd=float(np.std(v, ddof=1)) if len(v) > 1 else np.nan, min=float(np.min(v)), max=float(np.max(v)), values=[round(x, 4) for x in v]))
SEEDTAB = pd.DataFrame(SEEDTAB)
SEEDTAB.to_csv(os.path.join(TAB_DIR, 'table_seeds.csv'), index=False)
print(SEEDTAB.round(4).to_string(index=False))
if len(SEEDTAB) > 1 and SEEDTAB['n_seeds'].min() > 1:
    ES = SEEDTAB.set_index('arm')
    print('\nseed spread (max - min):', {a: round(float(ES.loc[a, 'max'] - ES.loc[a, 'min']), 4) for a in ES.index})
    for a, b in (('D_three_term', 'A_sup'), ('E_full', 'D_three_term')):
        if a in ES.index and b in ES.index:
            print(f'{a} - {b}: mean difference {ES.loc[a, "mean"] - ES.loc[b, "mean"]:+.4f}; lowest {a} seed {ES.loc[a, "min"]:.4f} vs highest {b} seed {ES.loc[b, "max"]:.4f} -> {"every seed of " + a + " beats every seed of " + b if ES.loc[a, "min"] > ES.loc[b, "max"] else "the seed ranges overlap"}')

### 10.3 The main model against its teachers: F1, calibration, thresholds
Platt maps (one slope and one intercept per label, fitted on fold 9, rare labels pooled) turn the raw logits into probabilities; the slope is kept positive so the ranking never changes. The calibration error is measured by cross-fitting (fit on one half of fold 9, score on the other). Thresholds are tuned for F1 on the calibrated probabilities.

In [ ]:
Zcr = LOG[CR]
CAL = {}
for n in ('DEAN', 'HOD committee', CR):
    a, b = fit_calibration(LOG[n], Y_va, CFG['min_pos'])
    P_ = sigmoid(LOG[n] * a + b)
    thr, f1 = tune_thresholds(Y_va, P_)
    raw_e, cal_e = eval_lib.crossfit_ece(LOG[n], Y_va, CFG['min_pos'])
    CAL[n] = dict(a=a, b=b, thr=thr, f1=f1, P=P_, ece_raw=raw_e, ece_cal=cal_e)
F1BOOT = {n: bootstrap(Y_va, CAL[n]['P'], CAL[n]['thr'], CFG['min_pos'], CFG['n_boot'], 0) for n in ('DEAN', 'HOD committee', CR)}
sumtab = pd.DataFrame({n: dict(macro_F1=F1BOOT[n]['macro_f1'][0], F1_lo=F1BOOT[n]['macro_f1'][1], F1_hi=F1BOOT[n]['macro_f1'][2],
                                ECE_raw=float(np.nanmean(CAL[n]['ece_raw'])), ECE_after_Platt=float(np.nanmean(CAL[n]['ece_cal']))) for n in CAL}).T
print(sumtab.round(4).to_string())
sumtab.to_csv(os.path.join(TAB_DIR, 'table_f1_calibration.csv'))
pl = per_label_table(LABELS, Y_va, CAL[CR]['P'], CAL[CR]['thr'], CFG['min_pos'])
pl['auc_dean'], pl['auc_hod'], pl['auc_prof'] = POINT['DEAN'], POINT['HOD committee'], POINT['Professors']
pl['auc_cr_lo'], pl['auc_cr_hi'] = np.nanpercentile(BOOT[CR], 2.5, 0), np.nanpercentile(BOOT[CR], 97.5, 0)
pl['reliable'] = REL
pl.to_csv(os.path.join(TAB_DIR, 'table_per_label.csv'), index=False)
print(pl.round(3).head(26).to_string(index=False))

## 11. What did the student take from each teacher, and what did Consultation do?
Four diagnostics, all on fold 9:

* **Fidelity** to each teacher (Spearman correlation and mean logit difference per label).
* **Gate mimicry**: the student's gate head against the cross-fitted routing distribution.
* **Relational fidelity**: the student's token Gram matrix against the cached DEAN Gram.
* **Consultation**: auxiliary path against standalone path, and where the attention goes. The auxiliary AUC on *training* records is measured too, because the cached Professor embeddings of a training record come from models that were fitted on that record's label (the in-sample problem of Stage 7); a large train-versus-validation gap of the auxiliary head would show that it is reading the label from them.

In [ ]:
net_main = RES[CR]['net'].to(dev).eval()
# fidelity
FID = []
for nm, cols, Zt in (('DEAN', np.arange(26), LOG['DEAN']), ('HOD committee', np.arange(26), LOG['HOD committee']), ('Professors', np.arange(5, 26), LOG['Professors'])):
    for j in cols:
        FID.append(dict(teacher=nm, label=LABELS[j], spearman=float(spearmanr(Zcr[:, j], Zt[:, j])[0]), mae=float(np.abs(Zcr[:, j] - Zt[:, j]).mean()), reliable=bool(REL[j])))
FID = pd.DataFrame(FID); FID.to_csv(os.path.join(TAB_DIR, 'table_fidelity.csv'), index=False)
print(FID[FID.reliable].groupby('teacher')[['spearman', 'mae']].mean().round(3).to_string())
# gate
Gv = PRED[CR]['val']['gate']
gprob = torch.softmax(torch.from_numpy(Gv), -1).numpy()
RV = G['RT_va']
GATE = dict(kl=float((RV * (np.log(RV) - np.log(gprob))).sum(1).mean()),
            top1_agree=float((gprob.argmax(1) == RV.argmax(1)).mean()),
            spearman=[float(spearmanr(gprob[:, k], RV[:, k])[0]) for k in range(5)],
            auc_student=auc_cols(Y_va[:, :5], gprob).tolist(), auc_nb03=auc_cols(Y_va[:, :5], RV).tolist())
print({k: (np.round(v, 3).tolist() if isinstance(v, list) else round(v, 4)) for k, v in GATE.items()})
# relational fidelity
@torch.no_grad()
def student_tokens(net, X, bs=256):
    out = []
    net.eval()
    for i in range(0, len(X), bs):
        with amp():
            o = net(X[i:i + bs])
        out.append(hod_lib.token_gram(o['tok']).cpu())
    return torch.cat(out).numpy()
GR_S = student_tokens(net_main, G['X_va'])
GR_T = CK['dean_gram_val'].astype(np.float32)
iu = np.triu_indices(N_TOK, 1)
REC_CORR = np.array([np.corrcoef(GR_S[i][iu], GR_T[i][iu])[0, 1] for i in range(NVA)])
RKD = dict(mse=float(((GR_S - GR_T) ** 2).mean()), baseline_mse_mean_gram=float(((GR_T.mean(0, keepdims=True) - GR_T) ** 2).mean()), corr_median=float(np.nanmedian(REC_CORR)))
print({k: round(v, 4) for k, v in RKD.items()})
check('relational fidelity: the student Gram is closer to the teacher Gram than the mean Gram is', RKD['mse'] < RKD['baseline_mse_mean_gram'], f"{RKD['mse']:.4f} vs {RKD['baseline_mse_mean_gram']:.4f}", level='warn')

In [ ]:
# Consultation: auxiliary against standalone, and attention
CONS = {}
if 'aux' in PRED[CR]['val']:
    Za_v, Za_t = PRED[CR]['val']['aux'], PRED[CR]['train']['aux']
    Zs_t = PRED[CR]['train']['logits']
    CONS = dict(aux_val=macro_auc_rel(Y_va, Za_v), standalone_val=macro_auc_rel(Y_va, Zcr), aux_train=macro_auc_rel(Y_tr, Za_t), standalone_train=macro_auc_rel(Y_tr, Zs_t),
                aux_logit_gap_val=float(np.abs(Za_v - Zcr).mean()))
    CONS['aux_gap'] = CONS['aux_train'] - CONS['aux_val']; CONS['standalone_gap'] = CONS['standalone_train'] - CONS['standalone_val']
    print({k: round(v, 4) for k, v in CONS.items()})
    att = PRED[CR]['val']['att'].mean(1)                       # (N,21) mean over heads
    ATT = np.zeros((21, 21))
    for j in range(21):
        pos = Y_va[:, 5 + j] > 0
        ATT[j] = att[pos].mean(0) if pos.sum() >= 3 else np.nan
    diag_adv = np.array([ATT[j, j] / np.nanmean(ATT[j]) for j in range(21)])
    CONS['attention_diag_advantage_median'] = float(np.nanmedian(diag_adv[REL_SUB]))
    CONS['attention_entropy_mean'] = float(-(att * np.log(att + 1e-9)).sum(1).mean())
    print(f'attention: median (weight on the matching Professor / mean weight) over reliable subclasses = {CONS["attention_diag_advantage_median"]:.2f} (1.0 = no preference); mean entropy {CONS["attention_entropy_mean"]:.2f} of ln 21 = {math.log(21):.2f}')
    check('Consultation: the auxiliary path is not worse than the standalone path on validation', CONS['aux_val'] >= CONS['standalone_val'] - 0.005, f"{CONS['aux_val']:.4f} vs {CONS['standalone_val']:.4f}", level='warn')

## 12. Figures
7.2 inches wide (two columns of a journal) or 3.5 inches (one column), 300 dpi PNG and vector PDF, Okabe-Ito colours. Family colours are the same in every figure.

In [ ]:
# Figure 1: the training graph and the deployed artefact
fig, ax = plt.subplots(figsize=(W2, 4.3)); ax.set_xlim(0, 100); ax.set_ylim(0, 60); ax.axis('off'); ax.grid(False)
def box(x, y, w, h, t, fc, fs=6.2, ec='#444444', ls='-', bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.25,rounding_size=1.0', fc=fc, ec=ec, lw=0.8, ls=ls))
    ax.text(x + w / 2, y + h / 2, t, ha='center', va='center', fontsize=fs, fontweight='bold' if bold else 'normal', linespacing=1.25)
def arr(x0, y0, x1, y1, c='#444444', ls='-'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', lw=0.8, color=c, ls=ls, shrinkA=0, shrinkB=0))
ax.add_patch(plt.Rectangle((1, 26), 56, 31, fc='#EAF3FA', ec='none')); ax.text(2, 55, 'DEPLOYED (kept)', fontsize=7, fontweight='bold', color=C_CR, va='center')
box(2, 40, 11, 10, 'ECG\n12 x 1000', '#F2F2F2', bold=True)
box(16, 40, 11, 10, 'Patchifier\nConv1d 25\n+ position', '#CFE3F3'); box(30, 40, 12, 10, 'ONE\nbidirectional\nMamba block', '#CFE3F3', bold=True); box(45, 40, 11, 10, 'LayerNorm\n+ mean pool\nz', '#CFE3F3')
arr(13, 45, 16, 45); arr(27, 45, 30, 45); arr(42, 45, 45, 45)
box(45, 28, 11, 8, 'Head\n5 + 21 logits', '#CFE3F3', bold=True); arr(50.5, 40, 50.5, 36)
ax.add_patch(plt.Rectangle((1, 1), 56, 23, fc='#FBF1E6', ec='none')); ax.text(2, 22, 'TRAINING ONLY (discarded)', fontsize=7, fontweight='bold', color=C_PROF, va='center')
box(4, 5, 13, 11, 'Cached Professor\nembeddings\n21 x %d' % P_DIM, '#F9D5BF', ls='--'); box(21, 5, 15, 11, 'Cross-attention\nConsultation\nquery = z', '#F9D5BF', ls='--', bold=True); box(40, 5, 14, 11, 'Auxiliary head\n26 logits', '#F9D5BF', ls='--')
arr(17, 10.5, 21, 10.5, ls='--'); arr(36, 10.5, 40, 10.5, ls='--'); arr(50.5, 40, 28.5, 16, c='#D55E00', ls='--')

# targets: each box says which loss it feeds; short arrows only, nothing crosses
ax.text(80, 57.5, 'cached targets (Stage 7)', fontsize=7, fontweight='bold', va='center')
tg = [('Professor + HOD logits (26)\n-> L_professor_KD, standalone logits', 47), ('cross-fitted routing g(x)\n-> L_gate_mimic, gate head', 36), ('DEAN token Gram 40 x 40\n-> L_RKD, student tokens', 25), ('labels\n-> L_supervised, both heads', 14)]
for t_, y_ in tg:
    box(66, y_, 33, 8, t_, '#F2F2F2', fs=5.8); arr(66, y_ + 4, 61, y_ + 4, c='#777777')
ax.add_patch(plt.Rectangle((60, 8), 1.2, 49, fc='#BBBBBB', ec='none'))
box(66, 3, 33, 8, 'Consultation auxiliary head\n-> labels and self-distillation to standalone', '#F9D5BF', fs=5.6, ls='--')
ax.text(1, -1.5, 'Only the blue part is exported. The standalone logits are computed without any Professor embedding; the smoke test verifies that they are bit-identical when the embeddings are replaced.', fontsize=6, style='italic', va='center')
ax.set_ylim(-3, 60)
save_fig(fig, 'fig01_architecture_training_vs_deployed')

In [ ]:
# Figure 2: parameter budget
hod_ref = hod_lib.HODNet(ENC_CFG, 4, 1, 8, D_TEACHER, 0.2, 256)
n_hod = sum(p.numel() for n, p in hod_ref.named_parameters() if not n.startswith('proj.')); del hod_ref
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.5), gridspec_kw=dict(width_ratios=[1.5, 1]))
ax = axs[0]
parts = [('patchifier + position', PARAMS['patchifier'], '#9ecae1'), ('Mamba block', PARAMS['mamba_block'], '#3182bd'), ('norm', PARAMS['norm'], '#c6dbef'), ('head', PARAMS['head'], '#08519c'),
         ('gate head', PARAMS['gate'], '#fdd0a2'), ('Consultation', PARAMS['consultation'], '#fd8d3c'), ('auxiliary head', PARAMS['aux_head'], '#d94801')]
left = 0
for nm, v, c in parts:
    ax.barh(0, v / 1e6, left=left, color=c, edgecolor='white', lw=0.5, height=0.5)
    if v / 1e6 > 0.07: ax.text(left + v / 2e6, 0, f'{nm}\n{v / 1e6:.2f}M', ha='center', va='center', fontsize=5.4, color='white' if c in ('#3182bd', '#08519c', '#d94801') else 'black')
    left += v / 1e6
ax.axvline(PARAMS['deployed'] / 1e6, color='k', lw=0.8, ls='--'); ax.text(PARAMS['deployed'] / 1e6, 0.42, 'deployed', ha='center', fontsize=6.3)
ax.set_yticks([]); ax.set_xlabel('parameters (millions)'); ax.set_title(f'Chief Resident, width {D_CR}'); panel(ax, 'a')
ax = axs[1]
names_ = ['DEAN\n(teacher)', 'one HOD\nhead', 'Chief\nResident']; vals_ = [T_PARAMS / 1e6, n_hod / 1e6, PARAMS['deployed'] / 1e6]
bars = ax.bar(names_, vals_, color=[C_DEAN, C_HOD, C_CR])
for b_, v in zip(bars, vals_): ax.text(b_.get_x() + b_.get_width() / 2, v * 1.02, f'{v:.2f}M', ha='center', fontsize=6.5)
ax.set_ylabel('deployed parameters (millions)'); ax.set_title(f'{T_PARAMS / PARAMS["deployed"]:.0f}x smaller than DEAN'); panel(ax, 'b')
fig.tight_layout(); save_fig(fig, 'fig02_parameter_budget')

In [ ]:
# Figure 3: the arms as a design matrix
cols_ = ['sup\n(BCE)', 'KD\nsuper', 'KD\nsub', 'gate', 'RKD', 'Consult.', 'self-\ndistil.']
def row_of(a):
    k = a['kd_src']; kd = a['w_kd'] > 0
    sup_src = {'chain': 'HOD', 'hod': 'HOD', 'dean_oof': 'DEAN\noof', 'dean_in': 'DEAN\nin', 'ens': 'mean'}[k] if kd else ''
    sub_src = {'chain': 'Prof.', 'hod': 'HOD', 'dean_oof': 'DEAN\noof', 'dean_in': 'DEAN\nin', 'ens': 'mean'}[k] if kd else ''
    return ['yes', sup_src, sub_src, 'yes' if a['w_gate'] > 0 else '', 'yes' if a['w_rkd'] > 0 else '', 'yes' if a['consult'] else '', 'yes' if a['w_sd'] > 0 else '']
fig, ax = plt.subplots(figsize=(W2, 3.0)); ax.axis('off'); ax.grid(False)
cell_c = {'yes': '#CFE3F3', 'HOD': '#9ecae1', 'Prof.': '#fdd0a2', 'DEAN\noof': '#bdbdbd', 'DEAN\nin': '#d9d9d9', 'mean': '#c7e9c0', '': 'white'}
tab = ax.table(cellText=[row_of(a) for a in ARMS.values()], rowLabels=list(ARMS), colLabels=cols_, cellColours=[[cell_c.get(c, 'white') for c in row_of(a)] for a in ARMS.values()], loc='center', cellLoc='center')
tab.auto_set_font_size(False); tab.set_fontsize(6.2); tab.scale(1, 1.45)
for (r, c), cell in tab.get_celld().items(): cell.set_linewidth(0.4)
ax.set_title('The ten arms: every row differs from the main model in exactly the cells that are blank or recoloured', pad=22)
save_fig(fig, 'fig03_arm_design_matrix')

In [ ]:
# Figure 4: learning-rate pilot
if isinstance(PILOT, pd.DataFrame) and len(PILOT):
    fig, axs = plt.subplots(1, 2, figsize=(W2, 2.4))
    ax = axs[0]; ax.plot(PILOT.lr, PILOT.val_macro_auc, 'o-', color=C_CR)
    for _, r_ in PILOT.iterrows(): ax.text(r_.lr, r_.val_macro_auc + 0.0015, f'{r_.val_macro_auc:.4f}' + ('' if r_.eligible else '\nskipped batches'), ha='center', fontsize=6)
    ax.axvline(LR, color='#555555', ls='--', lw=0.7); ax.set_xscale('log'); ax.set_xlabel('learning rate'); ax.set_ylabel('validation macro-AUC (%d ep.)' % CFG['pilot_epochs']); ax.set_title('Pilot'); panel(ax, 'a')
    ax = axs[1]
    for lr_ in CFG['pilot_lrs']:
        p_ = os.path.join(RUN_DIR, f'pilot_lr{lr_:g}_final.pt')
        if os.path.exists(p_):
            h = torch.load(p_, map_location='cpu', weights_only=False)['history']
            ax.plot([r_['epoch'] for r_ in h], [r_['val_macro_auc'] for r_ in h], marker='o', ms=2, label=f'{lr_:g}')
    ax.set_xlabel('epoch'); ax.set_ylabel('validation macro-AUC'); ax.legend(title='lr'); ax.set_title('Pilot curves'); panel(ax, 'b')
    fig.tight_layout(); save_fig(fig, 'fig04_lr_pilot')

In [ ]:
# Figure 5: training curves of the main model
H = pd.DataFrame(RES[CR]['history'])
fig, axs = plt.subplots(2, 3, figsize=(W2, 4.2))
ax = axs[0, 0]
for k, c in (('sup', '#0072B2'), ('kd', '#D55E00'), ('rkd', '#009E73'), ('gate', '#CC79A7'), ('aux', '#E69F00'), ('sd', '#56B4E9')):
    if H[k].abs().sum() > 0: ax.plot(H.epoch, H[k], label=k, color=c)
ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('training loss term'); ax.legend(ncol=2, fontsize=5.8); ax.set_title('Loss terms'); panel(ax, 'a')
ax = axs[0, 1]; ax.plot(H.epoch, H.val_macro_auc, color=C_CR, label='standalone (EMA)')
if 'val_aux_auc' in H: ax.plot(H.epoch, H.val_aux_auc, color=C_PROF, label='auxiliary (with Professors)')
for n_, c_ in (('DEAN', C_DEAN), ('HOD committee', C_HOD)): ax.axhline(macro(n_), color=c_, ls='--', lw=0.7, label=n_)
ax.axvline(RES[CR]['best_epoch'], color='#777777', ls=':', lw=0.7); ax.set_xlabel('epoch'); ax.set_ylabel('validation macro-AUC'); ax.legend(fontsize=5.6); ax.set_title('Validation AUC'); panel(ax, 'b')
ax = axs[0, 2]; ax.plot(H.epoch, H.val_logit_mae_chain, color='#D55E00'); ax.set_xlabel('epoch'); ax.set_ylabel('mean |student - cascade logit| (val)'); ax.set_title('Logit fidelity'); panel(ax, 'c')
ax = axs[1, 0]; ax.plot(H.epoch, H.val_gate_kl, color='#CC79A7'); ax.set_xlabel('epoch'); ax.set_ylabel('KL(route || gate), validation'); ax.set_title('Gate mimicry'); panel(ax, 'd')
ax = axs[1, 1]; ax.plot(H.epoch, H.lr, color='#555555'); ax.set_xlabel('epoch'); ax.set_ylabel('learning rate'); ax.set_title('Schedule'); panel(ax, 'e')
ax = axs[1, 2]
for t_, c_ in ((CR, C_CR), ('D_three_term_s0', '#009E73'), ('A_sup_s0', '#999999')):
    if t_ in RES: h_ = pd.DataFrame(RES[t_]['history']); ax.plot(h_.epoch, h_.val_macro_auc, color=c_, label=t_.rsplit('_s', 1)[0])
ax.set_xlabel('epoch'); ax.set_ylabel('validation macro-AUC'); ax.legend(fontsize=5.8); ax.set_title('Three arms'); panel(ax, 'f')
fig.tight_layout(); save_fig(fig, 'fig05_training_curves')

In [ ]:
# Figure 6: the arms on fold 9 (absolute and paired against supervised-only)
arm_tags = [f'{a}_s0' for a in ARMS if f'{a}_s0' in LOG]
order = sorted(arm_tags, key=lambda t: macro(t))
fig, axs = plt.subplots(1, 2, figsize=(W2, 3.3), gridspec_kw=dict(width_ratios=[1.15, 1]))
ax = axs[0]
for k, t in enumerate(order):
    v = macro(t, 'all', True); lo, hi = np.percentile(v, [2.5, 97.5]); c = C_CR if t == CR else '#7f7f7f'
    ax.plot([lo, hi], [k, k], color=c, lw=1.4); ax.plot(macro(t), k, 'o', color=c, ms=4)
ax.set_yticks(range(len(order))); ax.set_yticklabels([t.rsplit('_s', 1)[0] for t in order])
for n_, c_ in (('DEAN', C_DEAN), ('HOD committee', C_HOD)): ax.axvline(macro(n_), color=c_, ls='--', lw=0.8, label=n_)
ax.set_xlabel('validation macro-AUC (95 % bootstrap interval)'); ax.legend(loc='lower right'); ax.set_title('Absolute'); panel(ax, 'a')
ax = axs[1]
ref = S0('A_sup')
for k, t in enumerate(order):
    if t == ref: continue
    d = delta(t, ref); c = '#009E73' if d['lo'] > 0 else ('#D55E00' if d['hi'] < 0 else '#7f7f7f')
    ax.plot([d['lo'], d['hi']], [k, k], color=c, lw=1.4); ax.plot(d['delta'], k, 'o', color=c, ms=4)
ax.axvline(0, color='k', lw=0.7); ax.set_yticks(range(len(order))); ax.set_yticklabels([]); ax.set_xlabel('paired difference to supervised-only (macro-AUC)'); ax.set_title('Against A_sup'); panel(ax, 'b')
fig.tight_layout(); save_fig(fig, 'fig06_arms_absolute_and_paired')

In [ ]:
# Figure 7: the claims table as a forest plot
CLp = CLAIMS.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(W2, 0.27 * len(CLp) + 1.3))
vc = {'supported': '#009E73', 'not supported': '#7f7f7f', 'contradicted': '#D55E00'}
for k, r_ in CLp.iterrows():
    ax.plot([r_.lo, r_.hi], [k, k], color=vc[r_.verdict], lw=1.5); ax.plot(r_['delta'], k, 'o', color=vc[r_.verdict], ms=4)
    if r_.margin != 0: ax.plot([r_.margin, r_.margin], [k - 0.3, k + 0.3], color='k', lw=0.8)
ax.axvline(0, color='k', lw=0.7)
ax.set_yticks(range(len(CLp))); ax.set_yticklabels([f'{r_.id}  {r_.claim[:105]}' for _, r_ in CLp.iterrows()], fontsize=5.6)
for v, c in vc.items(): ax.plot([], [], 'o-', color=c, label=v)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3); ax.set_xlabel('paired difference of macro-AUC, a minus b (black tick: non-inferiority margin)'); ax.set_title('What the data support (fold 9)')
save_fig(fig, 'fig07_claims_forest')

In [ ]:
# Figure 8: seeds
fig, ax = plt.subplots(figsize=(W1 + 1.6, 2.7))
for k, a in enumerate(CFG['seed_arms']):
    vals = [macro(f'{a}_s{s}') for s in CFG['seeds'] if f'{a}_s{s}' in LOG]
    ax.plot([k] * len(vals), vals, 'o', color='#555555', ms=3.5, alpha=0.8)
    ax.plot([k - 0.18, k + 0.18], [np.mean(vals)] * 2, color=C_CR, lw=1.6)
    if len(vals) > 1: ax.text(k, max(vals) + 0.0006, f'sd {np.std(vals, ddof=1):.4f}', ha='center', fontsize=6)
for n_, c_ in (('DEAN', C_DEAN), ('HOD committee', C_HOD)): ax.axhline(macro(n_), color=c_, ls='--', lw=0.8, label=n_)
ax.set_xticks(range(len(CFG['seed_arms']))); ax.set_xticklabels(CFG['seed_arms']); ax.set_ylabel('validation macro-AUC'); ax.legend(); ax.set_title('Seed-to-seed spread', fontsize=8)
save_fig(fig, 'fig08_seeds')

In [ ]:
# Figure 9: per-label AUC
ordl = np.argsort([-(POINT[CR][j] if np.isfinite(POINT[CR][j]) else -1) for j in range(26)])
fig, ax = plt.subplots(figsize=(W2, 3.1)); x = np.arange(26)
ax.bar(x, POINT[CR][ordl], color=[SUPER_COL[LABELS[j] if j < 5 else FAMILY[LABELS[j]]] for j in ordl], alpha=0.85, width=0.7)
ax.errorbar(x, POINT[CR][ordl], yerr=[POINT[CR][ordl] - np.nanpercentile(BOOT[CR], 2.5, 0)[ordl], np.nanpercentile(BOOT[CR], 97.5, 0)[ordl] - POINT[CR][ordl]], fmt='none', ecolor='k', lw=0.6, capsize=1)
for nm, c, mk in (('DEAN', C_DEAN, '_'), ('HOD committee', C_HOD, 'x'), ('Professors', C_PROF, 'D')):
    v = np.where(AVAIL[nm], POINT[nm], np.nan)[ordl]; ax.plot(x, v, mk, color=c, ms=4.5 if mk != '_' else 7, mew=1.1, ls='none', label=nm)
ax.set_xticks(x); ax.set_xticklabels([LABELS[j] + ('' if REL[j] else '*') for j in ordl], rotation=60, ha='right')
for t, j in zip(ax.get_xticklabels(), ordl): t.set_color(SUPER_COL[LABELS[j] if j < 5 else FAMILY[LABELS[j]]])
ax.set_ylim(max(0.4, np.nanmin(POINT[CR]) - 0.05), 1.0); ax.set_ylabel('validation AUC'); ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.3)); ax.set_title('Chief Resident (bars, 95 % interval) against its teachers, per label (* fewer than 10 positives)')
save_fig(fig, 'fig09_per_label_auc')

In [ ]:
# Figure 10: paired differences against the teachers
fig, axs = plt.subplots(1, 3, figsize=(W2, 2.2), sharey=True)
for ax, sc in zip(axs, ('all', 'super', 'sub')):
    for k, n_ in enumerate(['DEAN', 'HOD committee', 'Professors']):
        if not mask_for([CR, n_], sc).any(): continue
        d = delta(CR, n_, sc); c = '#009E73' if d['lo'] > 0 else ('#D55E00' if d['hi'] < 0 else '#7f7f7f')
        ax.plot([d['lo'], d['hi']], [k, k], color=c, lw=1.6); ax.plot(d['delta'], k, 'o', color=c, ms=4.5)
    ax.axvline(0, color='k', lw=0.7); ax.axvspan(-0.01, 0.01, color='#cccccc', alpha=0.3, lw=0)
    ax.set_title({'all': 'all reliable labels', 'super': 'superclasses', 'sub': 'subclasses'}[sc]); ax.set_xlabel('macro-AUC, student - teacher')
axs[0].set_yticks(range(3)); axs[0].set_yticklabels(['DEAN', 'HOD committee', 'Professors'])
fig.suptitle('Paired differences to the teachers (grey band: +-0.01, the non-inferiority margin)', y=1.03, fontsize=8); fig.tight_layout(); save_fig(fig, 'fig10_paired_vs_teachers')

In [ ]:
# Figure 11: ROC and precision-recall of the five superclasses
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.9))
for j, s_ in enumerate(SUPER):
    fpr, tpr, _ = roc_curve(Y_va[:, j], CAL[CR]['P'][:, j]); axs[0].plot(fpr, tpr, color=SUPER_COL[s_], label=f'{s_} (AUC {POINT[CR][j]:.3f})')
    fpr2, tpr2, _ = roc_curve(Y_va[:, j], CAL['DEAN']['P'][:, j]); axs[0].plot(fpr2, tpr2, color=SUPER_COL[s_], lw=0.6, ls='--')
    pr, rc, _ = precision_recall_curve(Y_va[:, j], CAL[CR]['P'][:, j]); axs[1].plot(rc, pr, color=SUPER_COL[s_], label=f'{s_} (AP {average_precision_score(Y_va[:, j], CAL[CR]["P"][:, j]):.3f})')
    pr2, rc2, _ = precision_recall_curve(Y_va[:, j], CAL['DEAN']['P'][:, j]); axs[1].plot(rc2, pr2, color=SUPER_COL[s_], lw=0.6, ls='--')
axs[0].plot([0, 1], [0, 1], color='#999999', lw=0.5); axs[0].set_xlabel('false-positive rate'); axs[0].set_ylabel('true-positive rate'); axs[0].legend(loc='lower right'); axs[0].set_title('ROC (solid: Chief Resident, dashed: DEAN)'); panel(axs[0], 'a')
axs[1].set_xlabel('recall'); axs[1].set_ylabel('precision'); axs[1].legend(loc='lower left'); axs[1].set_title('Precision-recall'); panel(axs[1], 'b')
fig.tight_layout(); save_fig(fig, 'fig11_roc_pr_superclasses')

In [ ]:
# Figure 12: calibration
fig, axs = plt.subplots(1, 3, figsize=(W2, 2.5), gridspec_kw=dict(width_ratios=[1, 1, 1.35]))
rel_cols = np.where(REL)[0]
for ax, (ttl, getp) in zip(axs[:2], (('raw logits', lambda n: sigmoid(LOG[n])), ('after Platt (fold 9)', lambda n: CAL[n]['P']))):
    ax.plot([0, 1], [0, 1], color='#888888', lw=0.6, ls='--')
    for n_, c_ in ((CR, C_CR), ('DEAN', C_DEAN)):
        cf, ac, e = reliability(Y_va[:, rel_cols].ravel(), getp(n_)[:, rel_cols].ravel(), 12)
        ax.plot(cf, ac, 'o-', ms=2.5, color=c_, label=f'{"Chief Resident" if n_ == CR else n_} (ECE {e:.3f})')
    ax.set_xscale('symlog', linthresh=0.02); ax.set_yscale('symlog', linthresh=0.02); ax.set_xlabel('predicted probability'); ax.set_ylabel('observed frequency'); ax.legend(loc='upper left', fontsize=5.8); ax.set_title(ttl)
panel(axs[0], 'a'); panel(axs[1], 'b')
ax = axs[2]; x = np.arange(len(rel_cols)); w = 0.4
ax.bar(x - w / 2, CAL[CR]['ece_raw'][rel_cols], w, color='#9ecae1', label='raw'); ax.bar(x + w / 2, CAL[CR]['ece_cal'][rel_cols], w, color=C_CR, label='after Platt')
ax.set_xticks(x); ax.set_xticklabels([LABELS[j] for j in rel_cols], rotation=90, fontsize=5.5); ax.set_ylabel('cross-fitted ECE'); ax.legend(); ax.set_title('Per label'); panel(ax, 'c')
fig.tight_layout(); save_fig(fig, 'fig12_calibration')

In [ ]:
# Figure 13: F1 with tuned thresholds
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.9), gridspec_kw=dict(width_ratios=[2.2, 1]))
ax = axs[0]; x = np.arange(len(rel_cols)); w = 0.27
for k, (n_, c_) in enumerate(((CR, C_CR), ('HOD committee', C_HOD), ('DEAN', C_DEAN))):
    ax.bar(x + (k - 1) * w, CAL[n_]['f1'][rel_cols], w, color=c_, label='Chief Resident' if n_ == CR else n_)
ax.set_xticks(x); ax.set_xticklabels([LABELS[j] for j in rel_cols], rotation=60, ha='right'); ax.set_ylabel('F1 at the tuned threshold'); ax.legend(ncol=3, loc='upper right', fontsize=5.8); ax.set_title('F1 per label (reliable labels)'); panel(ax, 'a')
ax = axs[1]
ax.bar(range(3), [F1BOOT[n]['macro_f1'][0] for n in (CR, 'HOD committee', 'DEAN')], yerr=[[F1BOOT[n]['macro_f1'][0] - F1BOOT[n]['macro_f1'][1] for n in (CR, 'HOD committee', 'DEAN')], [F1BOOT[n]['macro_f1'][2] - F1BOOT[n]['macro_f1'][0] for n in (CR, 'HOD committee', 'DEAN')]],
       color=[C_CR, C_HOD, C_DEAN], capsize=2)
ax.set_xticks(range(3)); ax.set_xticklabels(['Chief\nResident', 'HOD', 'DEAN']); ax.set_ylabel('macro-F1 (95 % interval)'); ax.set_title('Macro'); panel(ax, 'b')
fig.tight_layout(); save_fig(fig, 'fig13_f1')

In [ ]:
# Figure 14: fidelity to each teacher
fig, axs = plt.subplots(1, 2, figsize=(W2, 2.9), gridspec_kw=dict(width_ratios=[2, 1]))
ax = axs[0]; Mf = np.full((3, 26), np.nan)
for i, t_ in enumerate(('DEAN', 'HOD committee', 'Professors')):
    d_ = FID[FID.teacher == t_]; Mf[i, [LABELS.index(l) for l in d_.label]] = d_.spearman.values
im = ax.imshow(Mf, vmin=0.3, vmax=1, cmap='viridis', aspect='auto'); ax.grid(False)
ax.set_xticks(range(26)); ax.set_xticklabels(LABELS, rotation=90, fontsize=5.8); ax.set_yticks(range(3)); ax.set_yticklabels(['DEAN', 'HOD', 'Professors'])
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.01); cb.set_label('Spearman r')
ax.set_title('How closely the student follows each teacher (validation logits)'); panel(ax, 'a')
ax = axs[1]; lab_ = ['DEAN', 'HOD committee', 'Professors']
ax.bar(range(3), [FID[(FID.teacher == t_) & FID.reliable].mae.mean() for t_ in lab_], color=[C_DEAN, C_HOD, C_PROF]); ax.set_xticks(range(3)); ax.set_xticklabels(['DEAN', 'HOD', 'Prof.']); ax.set_ylabel('mean |student - teacher| (logit)'); ax.set_title('Distance'); panel(ax, 'b')
fig.tight_layout(); save_fig(fig, 'fig14_fidelity_to_teachers')

In [ ]:
# Figure 15: gate mimicry
fig, axs = plt.subplots(1, 3, figsize=(W2, 2.5), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
for ax, (M_, ttl) in zip(axs[:2], ((np.array([RV[Y_va[:, c] > 0].mean(0) for c in range(5)]), 'cross-fitted gate g(x) (target)'), (np.array([gprob[Y_va[:, c] > 0].mean(0) for c in range(5)]), 'student gate head'))):
    im = ax.imshow(M_, cmap='Blues', vmin=0, vmax=0.6); ax.grid(False)
    ax.set_xticks(range(5)); ax.set_xticklabels(SUPER, fontsize=6); ax.set_yticks(range(5)); ax.set_yticklabels(SUPER, fontsize=6); ax.set_xlabel('routed to'); ax.set_ylabel('records positive for'); ax.set_title(ttl, fontsize=7)
    for i in range(5):
        for j in range(5): ax.text(j, i, f'{M_[i, j]:.2f}', ha='center', va='center', fontsize=6, color='white' if M_[i, j] > 0.35 else 'black')
panel(axs[0], 'a'); panel(axs[1], 'b')
ax = axs[2]
for j, s_ in enumerate(SUPER): ax.plot(RV[:, j], gprob[:, j], '.', ms=1.5, alpha=0.35, color=SUPER_COL[s_], label=f'{s_} (r {GATE["spearman"][j]:.2f})')
ax.plot([0, 1], [0, 1], color='k', lw=0.5); ax.set_xlabel('target g(x)'); ax.set_ylabel('student gate'); ax.legend(markerscale=5, fontsize=5.6, loc='upper left'); ax.set_title(f'KL {GATE["kl"]:.3f}, top-1 agreement {GATE["top1_agree"]:.2f}', fontsize=7); panel(ax, 'c')
fig.tight_layout(); save_fig(fig, 'fig15_gate_mimicry')

In [ ]:
# Figure 16: relational distillation
sel_ = [np.where(Y_va[:, c] > 0)[0][0] for c in (0, 1, 3) if (Y_va[:, c] > 0).any()]
fig = plt.figure(figsize=(W2, 4.8)); gs = fig.add_gridspec(3, len(sel_) + 1, width_ratios=[1] * len(sel_) + [1.3])
for k, i in enumerate(sel_):
    for r_, (M_, nm) in enumerate(((GR_T[i], 'DEAN (cached)'), (GR_S[i], 'Chief Resident'), (GR_S[i] - GR_T[i], 'difference'))):
        ax = fig.add_subplot(gs[r_, k]); ax.imshow(M_, vmin=-1, vmax=1, cmap='RdBu_r'); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
        if k == 0: ax.set_ylabel(nm, fontsize=6.5)
        if r_ == 0: ax.set_title(f'record {i}', fontsize=6.5)
ax = fig.add_subplot(gs[:2, -1]); ax.hist(REC_CORR[np.isfinite(REC_CORR)], 40, color=C_CR); ax.axvline(np.nanmedian(REC_CORR), color='k', ls='--', lw=0.7); ax.set_xlabel('per-record correlation of Gram entries'); ax.set_ylabel('validation records'); ax.set_title(f'median {np.nanmedian(REC_CORR):.2f}', fontsize=7); panel(ax, 'b')
ax = fig.add_subplot(gs[2, -1]); ax.bar(['student', 'mean Gram'], [RKD['mse'], RKD['baseline_mse_mean_gram']], color=[C_CR, '#999999']); ax.set_ylabel('MSE to the DEAN Gram'); panel(ax, 'c')
fig.suptitle('Relational distillation: the student reproduces how DEAN\'s 40 tokens relate to each other', y=0.99, fontsize=8); fig.tight_layout(); save_fig(fig, 'fig16_gram_relational')

In [ ]:
# Figure 17: Consultation
if CONS:
    fig, axs = plt.subplots(1, 3, figsize=(W2, 2.9), gridspec_kw=dict(width_ratios=[1.1, 1, 1.15]))
    ax = axs[0]; a_s = np.where(REL, POINT[CR], np.nan); a_a = np.where(REL, auc_cols(Y_va, PRED[CR]['val']['aux']), np.nan)
    ax.plot([0.5, 1], [0.5, 1], color='#888888', lw=0.6)
    for j in np.where(REL)[0]: ax.plot(a_s[j], a_a[j], 'o', color=SUPER_COL[LABELS[j] if j < 5 else FAMILY[LABELS[j]]], ms=3.5)
    ax.set_xlabel('standalone path AUC'); ax.set_ylabel('auxiliary path AUC (with Professors)'); ax.set_title('Per label, fold 9'); panel(ax, 'a')
    ax = axs[1]; w = 0.36
    ax.bar([0, 1], [CONS['standalone_train'], CONS['aux_train']], w, color='#fdd0a2', label='training records'); ax.bar([0 + w, 1 + w], [CONS['standalone_val'], CONS['aux_val']], w, color=C_CR, label='validation (fold 9)')
    lo_ = min(CONS['standalone_val'], CONS['aux_val'], CONS['standalone_train'], CONS['aux_train'])
    ax.set_ylim(max(0, lo_ - 0.05), 1.0); ax.set_xticks([w / 2, 1 + w / 2]); ax.set_xticklabels(['standalone', 'auxiliary']); ax.set_ylabel('macro-AUC'); ax.legend(loc='lower left', fontsize=5.6); ax.set_title(f'gap (train - val)\nstandalone {CONS["standalone_gap"]:+.3f}, aux. {CONS["aux_gap"]:+.3f}', fontsize=6.5); panel(ax, 'b')
    ax = axs[2]; im = ax.imshow(ATT, cmap='magma', aspect='auto'); ax.grid(False)
    ax.set_xticks(range(21)); ax.set_xticklabels(SUB, rotation=90, fontsize=5.2); ax.set_yticks(range(21)); ax.set_yticklabels(SUB, fontsize=5.2)
    for j in range(21): ax.plot(j, j, 's', mfc='none', mec='white', ms=3.2, mew=0.5)
    ax.set_xlabel('Professor attended'); ax.set_ylabel('records positive for'); cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label('mean attention'); ax.set_title(f'diagonal advantage {CONS["attention_diag_advantage_median"]:.2f}x', fontsize=7); panel(ax, 'c')
    fig.tight_layout(); save_fig(fig, 'fig17_consultation')

In [ ]:
# Figure 18: generalisation gap of every arm (in-sample targets and training-set sharpness)
gaps = []
for t in arm_tags:
    ztr = PRED[t]['train']['logits']
    tr_auc = float(np.nanmean(auc_per_label(Y_tr, ztr, 1)[REL])); gaps.append(dict(arm=t.rsplit('_s', 1)[0], train=tr_auc, val=macro(t)))
GAP = pd.DataFrame(gaps).set_index('arm'); GAP['gap'] = GAP.train - GAP.val; GAP.to_csv(os.path.join(TAB_DIR, 'table_generalisation_gap.csv'))
fig, ax = plt.subplots(figsize=(W2, 2.7)); x = np.arange(len(GAP)); w = 0.38
ax.bar(x - w / 2, GAP.train, w, color='#9ecae1', label='training records'); ax.bar(x + w / 2, GAP.val, w, color=C_CR, label='validation (fold 9)')
for i, (a_, r_) in enumerate(GAP.iterrows()): ax.text(i, max(r_.train, r_.val) + 0.002, f'{r_.gap:+.3f}', ha='center', fontsize=6)
ax.set_xticks(x); ax.set_xticklabels(GAP.index, rotation=35, ha='right'); ax.set_ylim(max(0, GAP[['train', 'val']].values.min() - 0.03), 1.0); ax.set_ylabel('macro-AUC'); ax.legend(ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0)); ax.set_title('Training against validation AUC of every arm (label shown: gap)', pad=16)
save_fig(fig, 'fig18_generalisation_gap')

In [ ]:
# Figure 19: hierarchy violations
PAR = np.array([LABELS.index(FAMILY[s]) for s in SUB])
def violations(Psub, Ppar): return float((Psub > Ppar + 0.05).mean())
calP = {n: CAL[n]['P'] for n in CAL}
a_p, b_p = fit_calibration(prof_va, Y_va[:, 5:], CFG['min_pos']); P_prof = sigmoid(prof_va * a_p + b_p)
VIO = {'Chief Resident': violations(calP[CR][:, 5:], calP[CR][:, PAR]), 'DEAN': violations(calP['DEAN'][:, 5:], calP['DEAN'][:, PAR]),
       'HOD committee': violations(calP['HOD committee'][:, 5:], calP['HOD committee'][:, PAR]), 'Professors (with HOD parents)': violations(P_prof, calP['HOD committee'][:, PAR])}
for t in ('A_sup_s0', 'D_three_term_s0'):
    if t in LOG:
        a_, b_ = fit_calibration(LOG[t], Y_va, CFG['min_pos']); P_ = sigmoid(LOG[t] * a_ + b_); VIO[t.rsplit('_s', 1)[0]] = violations(P_[:, 5:], P_[:, PAR])
fig, ax = plt.subplots(figsize=(W1 + 1.3, 2.7))
ks = list(VIO); ax.bar(range(len(ks)), [100 * VIO[k] for k in ks], color=[C_CR if k == 'Chief Resident' else '#999999' for k in ks])
for i, k in enumerate(ks): ax.text(i, 100 * VIO[k] + 0.01, f'{100 * VIO[k]:.2f}', ha='center', fontsize=6)
ax.set_xticks(range(len(ks))); ax.set_xticklabels([k.replace(' (', '\n(').replace(' ', '\n', 1) if len(k) > 9 else k for k in ks], fontsize=5.8); ax.set_ylabel('p(sub) > p(parent) + 0.05  (% of record x subclass)'); ax.set_title('Hierarchy consistency (calibrated)')
save_fig(fig, 'fig19_hierarchy_violations')

In [ ]:
# Figure 20: embedding space (t-SNE) of the student and of DEAN on fold 9
if CFG['run_tsne']:
    from sklearn.manifold import TSNE
    single = np.where(Y_va[:, :5].sum(1) == 1)[0]
    pick_t = single[np.random.default_rng(0).permutation(len(single))[:CFG['n_tsne']]]
    Zs = PRED[CR]['val']['z'][pick_t]; Zd = CK['dean_emb_val'][pick_t].astype(np.float32)
    lab_s = Y_va[pick_t, :5].argmax(1)
    fig, axs = plt.subplots(1, 2, figsize=(W2, 3.2))
    for ax, (Zx, ttl) in zip(axs, ((Zd, 'DEAN embedding (256-d)'), (Zs, f'Chief Resident embedding ({D_CR}-d)'))):
        Zx = (Zx - Zx.mean(0)) / (Zx.std(0) + 1e-6)
        E2 = TSNE(2, perplexity=min(30, max(5, len(pick_t) // 4)), init='pca', random_state=0).fit_transform(Zx)
        for c, s_ in enumerate(SUPER): ax.scatter(E2[lab_s == c, 0], E2[lab_s == c, 1], s=3, color=SUPER_COL[s_], label=s_, alpha=0.7, lw=0)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_title(ttl)
    axs[0].legend(markerscale=3, ncol=5, loc='lower center', bbox_to_anchor=(1.0, -0.12)); panel(axs[0], 'a'); panel(axs[1], 'b')
    fig.suptitle('Fold 9 records with exactly one superclass (t-SNE)', y=1.0, fontsize=8); fig.tight_layout(); save_fig(fig, 'fig20_embedding_tsne')

In [ ]:
# Figure 21: efficiency (parameters, latency) against accuracy
import time as _t
def bench(model, bs, reps, device):
    x = torch.randn(bs, 12, ENC_CFG['seq_len'], device=device); model = model.to(device).eval()
    def f():
        with torch.no_grad():
            return model(x)
    for _ in range(2): f()
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = _t.time()
    for _ in range(reps): f()
    if device.type == 'cuda': torch.cuda.synchronize()
    return (_t.time() - t0) / reps / bs * 1000
teacher_net, _tck = hod_lib.load_teacher(find_one('dean_finetuned.pt'), dev)
dep = ChiefResidentDeploy(ENC_CFG, D_CR, 5, 21, CFG['head_drop'], CFG['sub_hidden']).to(dev); dep.load_state_dict(net_main.deploy_state(), strict=True)
cpu = torch.device('cpu')
LAT = {}
for nm, m in (('DEAN', teacher_net), ('Chief Resident', dep)):
    LAT[nm] = dict(gpu_bs1=bench(m, 1, 10, dev), gpu_bs64=bench(m, 64, 3, dev), cpu_bs1=bench(copy.deepcopy(m).cpu(), 1, 3 if nm == 'DEAN' else 8, cpu))
dep.to(dev); del teacher_net
LAT = pd.DataFrame(LAT).T; LAT['params_M'] = [T_PARAMS / 1e6, PARAMS['deployed'] / 1e6]; LAT['macro_auc'] = [macro('DEAN'), macro(CR)]
LAT['fp16_size_MB'] = LAT.params_M * 2
LAT.to_csv(os.path.join(TAB_DIR, 'table_efficiency.csv')); print(LAT.round(3).to_string())
fig, axs = plt.subplots(1, 3, figsize=(W2, 2.5), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
ax = axs[0]
for n_, c_, mk in (('DEAN', C_DEAN, 's'), (CR, C_CR, 'o')):
    ax.plot(LAT.loc['DEAN' if n_ == 'DEAN' else 'Chief Resident', 'params_M'], macro(n_), mk, color=c_, ms=7, label='DEAN' if n_ == 'DEAN' else 'Chief Resident')
ax.plot(5 * n_hod / 1e6, macro('HOD committee'), 'D', color=C_HOD, ms=6, label='HOD committee (5 heads)'); ax.set_xscale('log'); ax.set_xlabel('deployed parameters (millions, log)'); ax.set_ylabel('validation macro-AUC'); ax.legend(loc='lower right'); ax.set_title('Accuracy against size'); panel(ax, 'a')
ax = axs[1]; x = np.arange(3); w = 0.38
for k, (n_, c_) in enumerate((('DEAN', C_DEAN), ('Chief Resident', C_CR))): ax.bar(x + (k - 0.5) * w, LAT.loc[n_, ['gpu_bs1', 'gpu_bs64', 'cpu_bs1']].values.astype(float), w, color=c_, label=n_)
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(['GPU\nbatch 1', 'GPU\nbatch 64', 'CPU\nbatch 1']); ax.set_ylabel('ms per record'); ax.legend(); ax.set_title('Latency'); panel(ax, 'b')
ax = axs[2]; sp = LAT.loc['DEAN', ['gpu_bs1', 'gpu_bs64', 'cpu_bs1']].values.astype(float) / LAT.loc['Chief Resident', ['gpu_bs1', 'gpu_bs64', 'cpu_bs1']].values.astype(float)
ax.bar(x, sp, color=C_CR); [ax.text(i, v * 1.02, f'{v:.1f}x', ha='center', fontsize=6.5) for i, v in enumerate(sp)]; ax.set_xticks(x); ax.set_xticklabels(['GPU\nbatch 1', 'GPU\nbatch 64', 'CPU\nbatch 1']); ax.set_ylabel('speed-up over DEAN'); ax.set_title(f'{T_PARAMS / PARAMS["deployed"]:.0f}x fewer parameters'); panel(ax, 'c')
fig.tight_layout(); save_fig(fig, 'fig21_efficiency')

In [ ]:
# Figure 22: which leads does the student use? (occlusion on fold 9)
OCC = None
if CFG['run_occlusion']:
    base = macro_auc_rel(Y_va, Zcr)
    occ = []
    for l in range(12):
        xo = G['X_va'].clone(); xo[:, l] = 0
        occ.append(base - macro_auc_rel(Y_va, predict(net_main, xo, G['PE_va'] if 'aux' in PRED[CR]['val'] else None)['logits']))
    OCC = pd.Series(occ, index=LEADS)
    fam_auc = np.zeros((12, 5))
    for l in range(12):
        xo = G['X_va'].clone(); xo[:, l] = 0
        zz = predict(net_main, xo, G['PE_va'] if 'aux' in PRED[CR]['val'] else None)['logits']
        fam_auc[l] = np.nan_to_num(auc_cols(Y_va, zz)[:5])
    base_s = np.nan_to_num(POINT[CR][:5])
    fig, axs = plt.subplots(1, 2, figsize=(W2, 2.6), gridspec_kw=dict(width_ratios=[1, 1.3]))
    ax = axs[0]; ax.bar(LEADS, OCC.values, color=C_CR); ax.tick_params(axis='x', labelsize=5.5); ax.set_ylabel('macro-AUC lost when the lead is zeroed'); ax.set_title('All reliable labels'); panel(ax, 'a')
    ax = axs[1]; im = ax.imshow((base_s[None, :] - fam_auc).T, cmap='Reds', aspect='auto'); ax.grid(False); ax.set_xticks(range(12)); ax.set_xticklabels(LEADS, fontsize=5.5); ax.set_yticks(range(5)); ax.set_yticklabels(SUPER)
    cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label('AUC lost'); ax.set_title('By superclass'); panel(ax, 'b')
    fig.tight_layout(); save_fig(fig, 'fig22_lead_occlusion')

## 13. Evidence table and gate
Every check of this notebook in one table. A **FAIL** stops the export; a **WARN** is something to read.

In [ ]:
CKT = pd.DataFrame(CHECKS); CKT.to_csv(os.path.join(TAB_DIR, 'table_checks.csv'), index=False)
with pd.option_context('display.max_colwidth', 120, 'display.width', 240):
    print(CKT.to_string(index=False))
n_fail = int((CKT.status == 'FAIL').sum())
print(f'\n{(CKT.status == "PASS").sum()} PASS | {(CKT.status == "WARN").sum()} WARN | {n_fail} FAIL')
if CFG['strict']:
    assert n_fail == 0, 'Failed checks: ' + '; '.join(CKT[CKT.status == 'FAIL'].check.tolist())

## 14. Export: the deployed artefact, and nothing else
`chief_resident_deploy.pt` holds the patchifier, the Mamba block and the standalone head (about 0.9 M parameters), the label order, the input normalisation, the Platt parameters and the F1 thresholds. The Consultation module, its auxiliary head and the gate head are not in it. The file is reloaded into a freshly built `ChiefResidentDeploy`, in fp32, and must reproduce the logits of the trained model.

In [ ]:
deploy_cfg = dict(ENC_CFG, d_model=D_CR, depth=1, grad_ckpt=False)
payload = dict(
    stage=8, arm='E_full', seed=0, width=D_CR, enc_cfg=deploy_cfg, head_drop=CFG['head_drop'], sub_hidden=CFG['sub_hidden'], n_sup=5, n_sub=21,
    deploy=net_main.deploy_state(), labels=LABELS, norm_mean=MU.tolist(), norm_std=SD.tolist(),
    platt_a=CAL[CR]['a'].tolist(), platt_b=CAL[CR]['b'].tolist(), thresholds=np.asarray(CAL[CR]['thr']).tolist(),
    n_params=int(PARAMS['deployed']), best_epoch=int(RES[CR]['best_epoch']), val_macro_auc=macro(CR), lr=LR, cfg={k: CFG[k] for k in SIG_KEYS},
    note='patchifier + one bidirectional Mamba block + standalone head. No Consultation, no auxiliary head, no gate head.')
torch.save(payload, os.path.join(WORK, 'chief_resident_deploy.pt'))
# reload from scratch, fp32
ck = torch.load(os.path.join(WORK, 'chief_resident_deploy.pt'), map_location='cpu', weights_only=False)
fresh = ChiefResidentDeploy(ck['enc_cfg'], ck['width'], ck['n_sup'], ck['n_sub'], ck['head_drop'], ck['sub_hidden']).to(dev)
fresh.load_state_dict(ck['deploy'], strict=True); fresh.eval(); net_main.eval()
check('deploy file: no training-only parameter', not any(k.startswith(cr_lib.TRAIN_ONLY_PREFIXES) for k in ck['deploy']))
check('deploy file: parameter count equals the deployed count of the trained model', sum(p.numel() for p in fresh.parameters()) == PARAMS['deployed'])
with torch.no_grad():
    xs_ = G['X_va'][:128]
    a_ = fresh(xs_).float().cpu().numpy(); b_ = net_main(xs_)['logits'].float().cpu().numpy()
check('deploy file reloaded from scratch reproduces the trained model (fp32, 128 records)', np.allclose(a_, b_, atol=1e-4), f'max |diff| {np.abs(a_ - b_).max():.2e}')
check('deploy file reproduces the evaluated validation logits (fp16 autocast tolerance)', np.abs(a_ - Zcr[:128]).max() < 0.1, f'max |diff| {np.abs(a_ - Zcr[:128]).max():.4f}', level='warn')
check('Platt parameters stored for all 26 labels', len(ck['platt_a']) == 26 and len(ck['thresholds']) == 26)
if CFG['strict']:
    assert all(c['status'] != 'FAIL' for c in CHECKS)

np.save(os.path.join(WORK, 'cr_logits_val.npy'), Zcr.astype(np.float32))
np.save(os.path.join(WORK, 'cr_logits_train.npy'), PRED[CR]['train']['logits'].astype(np.float32))
np.save(os.path.join(WORK, 'cr_emb_val.npy'), PRED[CR]['val']['z'].astype(np.float32))
np.save(os.path.join(WORK, 'cr_emb_train.npy'), PRED[CR]['train']['z'].astype(np.float32))
np.save(os.path.join(WORK, 'cr_gate_val.npy'), gprob.astype(np.float32))
json.dump({'a': CAL[CR]['a'].tolist(), 'b': CAL[CR]['b'].tolist(), 'thr': np.asarray(CAL[CR]['thr']).tolist(), 'labels': LABELS}, open(os.path.join(WORK, 'cr_calibration.json'), 'w'))
metrics = dict(
    width=D_CR, params=PARAMS, lr=LR, versions=VERSIONS, cfg=CFG, reference_teachers=TEACH.reset_index().to_dict('records'),
    macro_auc={n: {sc: macro(n, sc) for sc in ('all', 'super', 'sub') if mask_for([n], sc).any()} for n in LOG},
    main=dict(tag=CR, best_epoch=int(RES[CR]['best_epoch']), macro_f1=F1BOOT[CR]['macro_f1'], macro_auc_ci=ci(macro(CR, 'all', True))),
    claims=CLAIMS.to_dict('records'), seeds=SEEDTAB.to_dict('records'), gate=GATE, rkd=RKD, consultation=CONS, hierarchy_violations=VIO,
    generalisation_gap=GAP.reset_index().to_dict('records'), efficiency=LAT.reset_index().to_dict('records'),
    lead_occlusion=None if OCC is None else OCC.to_dict(), checks=CHECKS)
json.dump(metrics, open(os.path.join(WORK, 'cr_metrics.json'), 'w'), indent=1, default=float)
json.dump(CLAIMS.to_dict('records'), open(os.path.join(WORK, 'cr_claims.json'), 'w'), indent=1, default=float)
for f in glob.glob(os.path.join(RUN_DIR, '*_last.pt')):
    if os.path.exists(f.replace('_last.pt', '_final.pt')): os.remove(f)
shutil.rmtree(os.path.join(WORK, '__pycache__'), ignore_errors=True)
print(f'total notebook time {(time.time() - NB_T0) / 3600:.2f} h')
for root, _, files in os.walk(WORK):
    for f in sorted(files):
        if f.endswith('.pdf'): continue
        p = os.path.join(root, f)
        print(f'  {os.path.relpath(p, WORK):52s} {os.path.getsize(p) / 1e6:8.2f} MB')

## 15. The result, in words
The cell below writes a plain statement of what the data support, generated from the claims table so that the text cannot disagree with the numbers. It is a draft for the paper's results paragraph, not a replacement for reading the figures.

In [ ]:
def v(cid):
    r = CLAIMS[CLAIMS.id == cid]
    return None if r.empty else r.iloc[0]
lines = []
r = v('C13'); r2 = v('C14'); r3 = v('C16')
lines.append(f'The deployed Chief Resident has {PARAMS["deployed"] / 1e6:.2f} M parameters ({100 * PARAMS["deployed"] / T_PARAMS:.1f} % of DEAN) and reaches a validation macro-AUC of {macro(CR):.4f} '
             f'(95 % interval {ci(macro(CR, "all", True))[1]:.4f} to {ci(macro(CR, "all", True))[2]:.4f}) against {macro("DEAN"):.4f} for DEAN and {macro("HOD committee"):.4f} for the HOD committee.')
if r is not None:
    lines.append(f'Against DEAN the paired difference is {r["delta"]:+.4f} [{r["lo"]:+.4f}, {r["hi"]:+.4f}]: non-inferiority within 0.01 AUC is {r["verdict"]}' + (f'; superiority is {r3["verdict"]}.' if r3 is not None else '.'))
for cid, txt in (('C4', 'the three-term loss against supervised training'), ('C5', 'Consultation with self-distillation on top of the three-term loss'), ('C8', 'the cascade target against DEAN out-of-fold as the only teacher'),
                 ('C9', 'the cascade target against the departmental tier alone'), ('C11', 'the cascade target against the equal-weight ensemble')):
    rr = v(cid)
    if rr is not None: lines.append(f'{txt}: {rr["delta"]:+.4f} [{rr["lo"]:+.4f}, {rr["hi"]:+.4f}], {rr["verdict"]}.')
if CONS:
    lines.append(f'The auxiliary (Consultation) path reaches {CONS["aux_val"]:.4f} on validation against {CONS["standalone_val"]:.4f} for the standalone path; on training records the auxiliary path reaches {CONS["aux_train"]:.4f}, a gap of {CONS["aux_gap"]:+.3f} (standalone {CONS["standalone_gap"]:+.3f}).')
lines.append(f'The student reproduces the gate with KL {GATE["kl"]:.3f} and top-1 agreement {GATE["top1_agree"]:.2f}, and the DEAN token Gram with median per-record correlation {RKD["corr_median"]:.2f}.')
print('\n'.join('* ' + l for l in lines))
open(os.path.join(WORK, 'cr_results_draft.txt'), 'w').write('\n'.join(lines))

## 16. After this notebook
1. **Publish** `/kaggle/working` as the Dataset `chief-resident-v1`. Stage 9 (prototype index, retrieval-augmented inference) takes `chief_resident_deploy.pt`, `cr_emb_{train,val}.npy` and `cr_lib.py` from it.
2. **Read `table_claims.csv` first.** It decides what the paper may say.
   * If **C8 and C9 are supported** (the cascade beats DEAN out-of-fold and the departmental tier alone), the hierarchical claim stands and the Professor tier earns its cost.
   * If they are **not supported**, the honest statement is that distillation helps (C1, C4) but the hierarchy does not add measurably over a single good teacher, which is still a valid result (a 0.9 M model matching the 11.5 M teacher). Do not run more arms to find a significant one: the number of comparisons is already in the table.
   * If **C5 is not supported** but the auxiliary AUC on training records is far above its validation AUC (figure 17b), the Consultation module is reading the in-sample Professor embeddings; report the module as a training aid without measurable benefit and remove it from the headline method rather than tuning it.
3. **Fold 10** is evaluated once, in Stage 10, with `chief_resident_deploy.pt` and its stored Platt parameters and thresholds, with no further change to anything.
4. For the paper, `fig01` (architecture), `fig06`/`fig07` (ablation and claims), `fig09` (per label), `fig17` (Consultation) and `fig21` (efficiency) carry the argument; the others are supplementary.